## Transport in the Drygalski Basin 

### Goal:

- Estimate the transport of HSSW in the Drygalski Basin between mooring DITD and DITD. What is the along trough transport of HSSW in the Drygalski Basin? How does this transport change seasonally? 

### Steps: 

- Rotate velocity into along and cross trough direction (will require figuring out the orientation of the Drygalski Trough)

- Make some assumptions about the width of HSSW current or trough and vertical extent of HSSW to use as area for multipying velocity by to get volume transport 

- Use salt and heat flux equations to get salt and heat transport 


### Data Needs: 

- TNBD 1100 m sensor velocity and CTD (need to average to same sampling frequency)

- DITD 1100 m sensor velocity and CTD (need to average to same sampling frequency)

- GEBCO bathymetry data 

- CTD data to help estimate what is the vertical extent of HSSW layer 


### Tasks to get started on: 

- load in TNBD and DITD data 

- Estimate the orientation of Drygalski Trough based on latitude and longitude of mooring data 

- Rotate currents to along and across trough 

- Make preliminary plots of along and across trough currents 

- Make some crude assumptions about width of HSSW current and vertical extent to estimate volume flow through the basin

In [ ]:
# import packages 
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
from scipy import signal, interpolate
import gsw
import cmocean
from TNB_tools import trough_cross_section
import matplotlib.ticker as mticker

In [ ]:
def TS_template(ax1):
    """ 
    """
    # Create figure
    # fig = plt.figure(figsize=(10, 6),constrained_layout=True)
    # # Top left square
    # ax1 = plt.subplot(1,1,1)

    mint=-2.3
    maxt=-1.4
    mins=34.4
    maxs=36

    tempL=np.linspace(mint,maxt)
    salL=np.linspace(mins,maxs)
    Tg, Sg = np.meshgrid(tempL,salL)

    lon = 165
    lat = -75.0112
    p = 1093
    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    # rho = gsw.rho(SA, Tg, 0)
    # sigma_theta = rho - 1000

    # CALCULATE DENSITY (METHOD 2)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing1100 = gsw.t_freezing(gsw.SA_from_SP(salL, 0, lon, lat), p=0, saturation_fraction=0)
    plt.plot(salL, t_freezing1100, 'k--', label='Freezing Line at 0 dbar')

    cs = ax1.contour(Sg, Tg, sigma_theta, colors='k', zorder=1, alpha=0.7, levels=4)
    cl=plt.clabel(cs,fontsize=10,inline=True,manual=[(34.45,-2.2),(34.55,-2.05),(34.75,-1.75)])

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal, interpolate

# ── 1. GEOMETRIC TROUGH ORIENTATION ─────────────────────────────────────────
def trough_orientation_from_moorings(lon1, lat1, lon2, lat2):
    """
    Estimate trough axis orientation from two mooring positions.
    
    Assumes the two along-trough moorings define the trough axis.
    Uses a local flat-Earth approximation (valid over short distances
    typical of continental shelf troughs, ~10s of km).
    
    Parameters
    ----------
    lon1, lat1 : float
        Position of mooring 1 (decimal degrees)
    lon2, lat2 : float
        Position of mooring 2 (decimal degrees)
    
    Returns
    -------
    theta_deg : float
        Angle of trough axis CCW from East (degrees)
        This is the angle you rotate velocities BY to get along-trough
    bearing_deg : float
        Geographic bearing from mooring 1 to mooring 2 (degrees CW from North)
        Useful for sanity-checking against a map
    dx, dy : float
        Approximate along-East and along-North separation (km)
    """
    # Mean latitude for scaling (flat-Earth approximation)
    lat_mean = np.deg2rad((lat1 + lat2) / 2.0)
    
    R_earth = 6371.0  # km
    
    # Zonal and meridional separation
    dx = R_earth * np.cos(lat_mean) * np.deg2rad(lon2 - lon1)  # km, East+
    dy = R_earth * np.deg2rad(lat2 - lat1)                      # km, North+
    
    # Angle CCW from East — this is theta for the rotation matrix
    theta_deg = np.rad2deg(np.arctan2(dy, dx))
    
    # Geographic bearing (CW from North) — easier to sanity-check on a map
    bearing_deg = np.rad2deg(np.arctan2(dx, dy)) % 360
    
    dist_km = np.sqrt(dx**2 + dy**2)
    
    print(f"Mooring separation:  {dist_km:.1f} km")
    print(f"dx (East):           {dx:.1f} km")
    print(f"dy (North):          {dy:.1f} km")
    print(f"Trough axis angle from East (theta): {theta_deg:.1f}°")
    print(f"Geographic bearing M1→M2:            {bearing_deg:.1f}° (CW from North)")
    print(f"\nSanity check: positive v_along will point toward M2")
    
    return theta_deg, bearing_deg, dx, dy


# ── 2. VELOCITY ROTATION ─────────────────────────────────────────────────────
# def rotate_velocities(u, v, theta_deg):
#     """
#     Rotate (u=East, v=North) to (along-trough, across-trough).
    
#     Sign convention: positive v_along points from mooring 1 toward mooring 2.
#     Flip sign afterward if that's up-trough rather than toward shelf break.
    
#     Parameters
#     ----------
#     u, v      : array-like, eastward and northward velocity (m/s)
#     theta_deg : float, trough axis angle CCW from East
    
#     Returns
#     -------
#     v_along, v_across : rotated velocity components (m/s)
#     """
#     theta = np.deg2rad(theta_deg)
#     v_along  =  u * np.cos(theta) - v * np.sin(theta) # u_prime, positive towards shelf break 
#     v_across = u * np.sin(theta) + v * np.cos(theta) # v_prime, positive towards land (NW direction)
#     return v_along, v_across

def rotate_velocities(u, v, theta_deg):
    """
    Rotate (u=East, v=North) to (along-trough, across-trough).
    
    Sign convention: positive v_along points from mooring 1 toward mooring 2.
    Flip sign afterward if that's up-trough rather than toward shelf break.
    
    Parameters
    ----------
    u, v      : array-like, eastward and northward velocity (m/s)
    theta_deg : float, trough axis angle CCW from East
    
    Returns
    -------
    v_along, v_across : rotated velocity components (m/s)
    """
    theta = np.deg2rad(theta_deg)
    v_along  =  u * np.cos(theta) + v * np.sin(theta) # u_prime, positive towards shelf break 
    v_across = -u * np.sin(theta) + v * np.cos(theta) # v_prime, positive towards land (NW direction)
    return v_along, v_across



# ── 3. CROSS-SECTIONAL AREA ──────────────────────────────────────────────────
def estimate_cross_section(trough_widths_m, depth_bins_m, hssw_layer_top_m,
                            hssw_layer_bottom_m):
    """
    Estimate the cross-sectional area occupied by HSSW.
    
    For a single velocity measurement, this encodes your depth assumption.
    
    Parameters
    ----------
    trough_widths_m      : array of trough widths at each depth bin (m)
                           Get this from IBCSO/BedMachine bathymetry transect
    depth_bins_m         : array of depths corresponding to trough_widths_m (m)
    hssw_layer_top_m     : shallowest extent of HSSW layer (m, positive down)
                           Estimate from CTD casts or climatology
    hssw_layer_bottom_m  : deepest extent — usually trough floor at mooring loc
    
    Returns
    -------
    A_total    : total cross-sectional area of HSSW layer (m²)
    A_simple   : simple rectangular estimate W * H (m²) for comparison
    W_at_instr : trough width at instrument depth (m)
    """
    # Integrate trough width over the HSSW depth range
    mask = (depth_bins_m >= hssw_layer_top_m) & (depth_bins_m <= hssw_layer_bottom_m)
    
    if mask.sum() < 2:
        raise ValueError("Not enough depth bins within HSSW layer — "
                         "check your depth arrays")
    
    A_total = np.trapz(trough_widths_m[mask], depth_bins_m[mask])
    
    # Simple rectangular estimate (what you'd use with a single width value)
    H = hssw_layer_bottom_m - hssw_layer_top_m
    W_at_instr = float(np.interp(1100.0, depth_bins_m, trough_widths_m))
    A_simple = W_at_instr * H
    
    print(f"HSSW layer thickness:         {H:.0f} m")
    print(f"Trough width at 1100m:        {W_at_instr/1000:.1f} km")
    print(f"Area (integrated over width): {A_total/1e6:.2f} km²")
    print(f"Area (simple rectangle):      {A_simple/1e6:.2f} km²")
    print(f"Difference:                   {100*(A_total-A_simple)/A_simple:.1f}%")
    
    return A_total, A_simple, W_at_instr


def area_uncertainty(A_best, layer_thickness_uncertainty_m, W_at_instr):
    """
    Propagate layer thickness uncertainty into area and transport uncertainty.
    
    A rough ±1-sigma bound assuming your main uncertainty is in layer thickness.
    """
    A_low  = W_at_instr * (A_best / W_at_instr - layer_thickness_uncertainty_m)
    A_high = W_at_instr * (A_best / W_at_instr + layer_thickness_uncertainty_m)
    print(f"Area range: {A_low/1e6:.2f} – {A_high/1e6:.2f} km²")
    return A_low, A_high


# ── 4. FILTERING ─────────────────────────────────────────────────────────────
def lowpass_filter(data, cutoff_hours=36, dt_hours=1.0):
    """
    Zero-phase Butterworth low-pass filter.
    
    36h cutoff removes semidiurnal/diurnal tides and near-inertial motions.
    Inertial period at 70°S ~ 12.9h, so 36h is conservative.
    Consider 72h or 120h if you want to focus on synoptic/seasonal variability.
    """
    nyq = 0.5 / dt_hours          # Nyquist in cycles/hour
    fc  = 1.0 / cutoff_hours      # cutoff in cycles/hour
    b, a = signal.butter(4, fc / nyq, btype='low')
    # Handle NaNs by interpolating over gaps before filtering
    t = np.arange(len(data))
    finite = np.isfinite(data)
    data_filled = np.interp(t, t[finite], data[finite])
    filtered = signal.filtfilt(b, a, data_filled)
    filtered[~finite] = np.nan    # restore NaN gaps
    return filtered


# ── 5. HSSW IDENTIFICATION ───────────────────────────────────────────────────
def hssw_mask(salinity, temperature, S_threshold=34.6, T_threshold=-1.7):
    """
    Boolean mask for HSSW presence at the instrument.
    
    Adjust thresholds to your specific region — HSSW definitions vary
    between Ross Sea (~34.75), Weddell (~34.6), and other shelf troughs.
    Cross-reference against your CTD-based T/S diagram.
    """
    return (salinity >= S_threshold) & (temperature <= T_threshold)


# ── 6. TRANSPORT ─────────────────────────────────────────────────────────────
def compute_transports(v_along, salinity, temperature,
                       A, rho=1027.0, cp=3985.0, T_ref=-1.9):
    """
    Compute volume, salt, and heat transport time series.
    
    Parameters
    ----------
    v_along     : along-trough velocity (m/s), already low-pass filtered
    salinity    : practical salinity (g/kg)
    temperature : in-situ or potential temperature (°C)
    A           : cross-sectional area (m²)
    rho         : reference density (kg/m³) — ideally compute from T/S
    cp          : specific heat capacity (J/kg/K)
    T_ref       : reference temperature (°C) for heat transport
                  Use surface freezing point (~-1.9°C) for shelf work
    
    Returns
    -------
    dict of transport arrays, all as time series
    """
    Q   = v_along * A                              # m³/s
    F_S = rho * v_along * salinity * A             # kg/s
    F_H = rho * cp * v_along * (temperature - T_ref) * A  # W
    
    return {
        'Q_m3s':         Q,
        'Q_Sv':          Q / 1e6,
        'salt_flux_kgs': F_S,
        'heat_flux_W':   F_H,
        'heat_flux_GW':  F_H / 1e9,
    }


# ── 7. EDDY DECOMPOSITION ────────────────────────────────────────────────────
def eddy_flux_decomposition(v_along, scalar, A, rho=1027.0):
    """
    Decompose total flux into mean and eddy (turbulent) components.
    
    Total flux  = <v> * <S> * A  +  <v'S'> * A
    Mean flux   = v_bar * S_bar * A
    Eddy flux   = <v'S'> * A     (often non-negligible in troughs!)
    
    Works for salinity or temperature as the scalar.
    """
    v_bar = np.nanmean(v_along)
    s_bar = np.nanmean(scalar)
    
    v_prime = v_along - v_bar
    s_prime = scalar  - s_bar
    
    eddy_covariance = np.nanmean(v_prime * s_prime)
    
    mean_flux = rho * v_bar  * s_bar          * A
    eddy_flux = rho * eddy_covariance         * A
    total     = rho * np.nanmean(v_along * scalar) * A
    
    print(f"Mean flux:  {mean_flux:.2e}")
    print(f"Eddy flux:  {eddy_flux:.2e}  ({100*eddy_flux/total:.1f}% of total)")
    print(f"Total flux: {total:.2e}")
    
    return mean_flux, eddy_flux, total


# ── 8. MAIN WORKFLOW ─────────────────────────────────────────────────────────
def full_transport_pipeline(ds,
                             lon1, lat1, lon2, lat2,
                             trough_widths_m, depth_bins_m,
                             hssw_top_m, hssw_bottom_m,
                             S_thresh=34.6, T_thresh=-1.7,
                             T_ref=-1.9, dt_hours=1.0):
    """
    ds : xarray Dataset with variables u, v, temp, salinity (dim: time)
    lon1/lat1, lon2/lat2 : mooring positions (decimal degrees)
    trough_widths_m : trough width profile from bathymetry (m)
    depth_bins_m    : corresponding depths (m, positive down)
    hssw_top_m      : top of HSSW layer (m)
    hssw_bottom_m   : bottom of HSSW layer / trough floor (m)
    """
    # 1. Orientation
    print("=== TROUGH ORIENTATION ===")
    theta_geo, bearing, dx, dy = trough_orientation_from_moorings(
        lon1, lat1, lon2, lat2)
    theta_pca = principal_axis_angle(ds.u.values, ds.v.values)
    print(f"\nPCA angle cross-check: {theta_pca:.1f}°  "
          f"(difference: {abs(theta_geo - theta_pca):.1f}°)")
    print("If difference > 15°, examine whether flow is truly channelized\n")
    
    # Use geometric angle as primary; flag if PCA disagrees strongly
    theta = theta_geo
    
    # 2. Rotate
    v_along, v_across = rotate_velocities(ds.u.values, ds.v.values, theta)
    v_along_lp = lowpass_filter(v_along, cutoff_hours=36, dt_hours=dt_hours)
    
    # 3. Cross-sectional area
    print("=== CROSS-SECTIONAL AREA ===")
    A_integrated, A_simple, W_instr = estimate_cross_section(
        trough_widths_m, depth_bins_m, hssw_top_m, hssw_bottom_m)
    
    # 4. Transports (best estimate + uncertainty bounds)
    print("\n=== TRANSPORT (best estimate) ===")
    T_best = compute_transports(v_along_lp, ds.salinity.values,
                                 ds.temp.values, A_integrated)
    
    # 5. HSSW mask
    mask = hssw_mask(ds.salinity.values, ds.temp.values, S_thresh, T_thresh)
    print(f"\nHSSW present {100*mask.mean():.1f}% of record")
    
    # 6. Build output DataFrame
    df = pd.DataFrame({
        'v_along_raw':   v_along,
        'v_along_lp':    v_along_lp,
        'v_across':      v_across,
        'is_HSSW':       mask,
        **T_best
    }, index=ds.time.values)
    
    return df, theta_geo, A_integrated

### Load in data 

In [ ]:
# # DELL 
# bath = xr.open_dataset('/Users/mblan/Documents/research/data/gebco_2024_n-66.2651_s-87.7985_w159.418_e213.2403.nc')

# # DELL BATHYMETRY - BED MACHINE 

bath = xr.open_dataset('data/BedMachine_TNB_subset.nc')

# # MAC
# bath = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/gebco_2024_n-66.2651_s-87.7985_w159.418_e213.2403.nc')

In [ ]:
# MAC
# moorings = xr.open_datatree('/Users/mackenzie/Documents/Research/PhD/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')
# # Load in data - MAC 
# TNBD_ctd_top = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_ctd_top_999.nc')
# TNBD_ctd_bot = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_ctd_bot_1093.nc')

# TNBD_vel_top = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_vel_top_999.nc')
# TNBD_vel_bot = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_vel_bot_1093.nc')

# DITD_ctd_top = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/DITD_2018_ctd_top.nc')
# DITD_ctd_bot = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/DITD_2018_ctd_bot.nc')

# DITD_vel_top = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/vel_1100.nc')
# DITD_vel_bot = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/vel_1200.nc')

In [ ]:
# LOAD IN SAVED INTERMEDIATE FILES - MAC 
# 30 min averaged data 

TNBD_ctd_top = xr.open_dataset('data/TNBD_ctd_top_30min.nc')
TNBD_ctd_bot = xr.open_dataset('data/TNBD_ctd_bot_30min.nc')
TNBD_vel_top = xr.open_dataset('data/TNBD_vel_top_30min.nc')
TNBD_vel_bot = xr.open_dataset('data/TNBD_vel_bot_30min.nc')

DITD_ctd_top = xr.open_dataset('data/DITD_ctd_top_30min.nc')
DITD_ctd_bot = xr.open_dataset('data/DITD_ctd_bot_30min.nc')
DITD_vel_top = xr.open_dataset('data/DITD_vel_top_30min.nc')
DITD_vel_bot = xr.open_dataset('data/DITD_vel_bot_30min.nc')      # sensor cut out in june 2018

D_ctd_top = xr.open_dataset('data/D_ctd_top_30min.nc')
D_ctd_mid = xr.open_dataset('data/D_ctd_mid_30min.nc')
# D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
D_vel_mid = xr.open_dataset('data/D_vel_mid_30min.nc')
D_vel_bot = xr.open_dataset('data/D_vel_bot_30min.nc')

In [ ]:
# # load in data - DELL 
# TNBD_ctd_bot = xr.open_dataset("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2018_19 Araon Cruise (ANA09B) - NIWAMooringRetrieval/Data/TNBD/working/tnbd_2018_ctd_bot_1093.nc")
# TNBD_vel_bot = xr.open_dataset("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2018_19 Araon Cruise (ANA09B) - NIWAMooringRetrieval/Data/TNBD/working/tnbd_2018_vel_bot_1093.nc")



# DITD_ctd_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_top.nc')
# DITD_vel_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1100.nc')


# moorings = xr.open_datatree('/Users/mblan/Documents/research/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

In [ ]:
# TNBD_ctd_bot = moorings['TNBD/2018/ctd/bot'].to_dataset()
# TNBD_vel_bot = moorings['TNBD/2018/vel/bot'].to_dataset()
# TNBD_ctd_top = moorings['TNBD/2018/ctd/top'].to_dataset()
# TNBD_vel_top = moorings['TNBD/2018/vel/top'].to_dataset()

# DITD_ctd_bot = moorings['DITD/2018/ctd/bot'].to_dataset()
# DITD_vel_bot = moorings['DITD/2018/vel/bot'].to_dataset()
# DITD_ctd_top = moorings['DITD/2018/ctd/top'].to_dataset()
# DITD_vel_top = moorings['DITD/2018/vel/top'].to_dataset()


In [ ]:
%matplotlib widget

In [ ]:
# fix DITD time - should save this change permenantly later
# DITD_vel_top['time'] = DITD_vel_top.time - pd.Timedelta(12,'h')

In [ ]:
# rename variables in DITD velocity dataset 
# DITD_vel_top = DITD_vel_top.rename({
#     'east':'u',
#     'north':'v',
#     'up':'w',
#     'pressure':'pres',
#     'direction':'dir'
# }).assign_coords({
#     'z':1121,
#     'lat':-75.28,
#     'lon':164.1
# })
# # rename variables in DITD velocity dataset 
# DITD_vel_bot = DITD_vel_bot.rename({
#     'east':'u',
#     'north':'v',
#     'up':'w',
#     'pressure':'pres',
#     'direction':'dir'
# }).assign_coords({
#     'z':1218,
#     'lat':-75.28,
#     'lon':164.1
# })

In [ ]:
# D_vel_bot = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_vel_mid = moorings['D/2017/vel/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_ctd_mid = moorings['D/2017/ctd/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))

In [ ]:
# D_vel_top = moorings['D/2017/vel/top'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_vel_bot = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_vel_mid = moorings['D/2017/vel/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))

# D_ctd_mid = moorings['D/2017/ctd/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_ctd_top = moorings['D/2017/ctd/top'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
# D_ctd_bot = moorings['D/2017/ctd/bot'].to_dataset()

In [ ]:
# convert velocity from m/s to cm/s 
# DITD_vel_top['u'] = DITD_vel_top.u * 100
# DITD_vel_top['v'] = DITD_vel_top.v * 100
# DITD_vel_top['speed'] = DITD_vel_top.speed * 100

# DITD_vel_bot['v'] = DITD_vel_bot.v * 100
# DITD_vel_bot['u'] = DITD_vel_bot.u * 100
# DITD_vel_bot['speed'] = DITD_vel_bot.speed * 100


### Identify trough orientation based on mooring positions

In [ ]:
tnbd_lat = -75.0112
tnbd_lon = 165.5562

ditd_lat = -75.2751
ditd_lon = 164.0625

In [ ]:
# using great circle distance 
tnbd_lat_rad = np.deg2rad(tnbd_lat)
tnbd_lon_rad = np.deg2rad(tnbd_lon)
ditd_lat_rad = np.deg2rad(ditd_lat)
ditd_lon_rad = np.deg2rad(ditd_lon)

In [ ]:
delta_phi = ditd_lat_rad - tnbd_lat_rad
delta_y = ditd_lon_rad - tnbd_lon_rad

In [ ]:
a = np.sin(delta_phi/2)**2 + np.cos(ditd_lat_rad)*np.cos(tnbd_lat_rad)*np.sin(delta_y/2)**2

In [ ]:
c = 2*np.arctan2(np.sqrt(a),np.sqrt(1-a))

In [ ]:
R = 6371 #km
d = R*c

In [ ]:
d

In [ ]:
theta = np.arctan2((np.sin(delta_y)*np.cos(tnbd_lat_rad)),(np.cos(ditd_lat_rad)*np.sin(tnbd_lat_rad)-np.sin(ditd_lat_rad)*np.cos(tnbd_lat_rad)*np.cos(delta_y)))

In [ ]:
theta

In [ ]:
np.rad2deg(theta)

In [ ]:
theta_deg, bearing_deg, dx, dy = trough_orientation_from_moorings(lon1=ditd_lon, lat1=ditd_lat, lon2=tnbd_lon, lat2=tnbd_lat)

In [ ]:
theta_deg

In [ ]:
90+np.rad2deg(theta)

### Resample velocity and ctd data to be same sampling frequency

In [ ]:
# # convert velocity to m/s for all moorings 
# TNBD_vel_bot['v'] = TNBD_vel_bot.v / 100 
# TNBD_vel_bot['u'] = TNBD_vel_bot.u / 100 

# TNBD_vel_top['v'] = TNBD_vel_top.v / 100 
# TNBD_vel_top['u'] = TNBD_vel_top.u / 100 

# DITD_vel_bot['v'] = DITD_vel_bot.v / 100 
# DITD_vel_bot['u'] = DITD_vel_bot.u / 100 

# DITD_vel_top['v'] = DITD_vel_top.v / 100 
# DITD_vel_top['u'] = DITD_vel_top.u / 100 

# D_vel_bot['v'] = D_vel_bot.v / 100 
# D_vel_bot['u'] = D_vel_bot.u / 100 

# D_vel_mid['v'] = D_vel_mid.v / 100 
# D_vel_mid['u'] = D_vel_mid.u / 100 



In [ ]:
# # need to get all datasets on same time and sampling interval 

# time_slice = slice('2018-03-14','2018-12-31')
# resample = '30min'

In [ ]:
# TNBD_ctd_top = TNBD_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# TNBD_ctd_bot = TNBD_ctd_bot.sel(time=time_slice).resample(time=resample).mean()
# TNBD_vel_top = TNBD_vel_top.sel(time=time_slice).resample(time=resample).mean()
# TNBD_vel_bot = TNBD_vel_bot.sel(time=time_slice).resample(time=resample).mean()

# DITD_ctd_top = DITD_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# DITD_ctd_bot = DITD_ctd_bot.sel(time=time_slice).resample(time=resample).mean()
# DITD_vel_top = DITD_vel_top.sel(time=time_slice).resample(time=resample).mean()
# DITD_vel_bot = DITD_vel_bot.sel(time=slice('2018-03-14','2018-06')).resample(time=resample).mean()       # sensor cut out in june 2018

# D_ctd_top = D_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# D_ctd_mid = D_ctd_mid.sel(time=time_slice).resample(time=resample).mean()
# # D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# # D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
# D_vel_mid = D_vel_mid.sel(time=time_slice).resample(time=resample).mean()
# D_vel_bot = D_vel_bot.sel(time=time_slice).resample(time=resample).mean()

In [ ]:
# TNBD_ctd_top['abs_sal'] = gsw.SA_from_SP(TNBD_ctd_top.sal,1000,TNBD_ctd_top.lon,TNBD_ctd_top.lat)

In [ ]:
# # TNBD_ctd_bot['abs_sal'] = gsw.SA_from_SP(TNBD_ctd_bot.sal,1100,TNBD_ctd_bot.lon,TNBD_ctd_bot.lat)
# DITD_ctd_bot['abs_sal'] = gsw.SA_from_SP(DITD_ctd_bot.sal,1200,DITD_ctd_bot.lon,DITD_ctd_bot.lat)
# DITD_ctd_top['abs_sal'] = gsw.SA_from_SP(DITD_ctd_top.sal,1100,DITD_ctd_top.lon,DITD_ctd_top.lat)

# D_ctd_top['abs_sal'] = gsw.SA_from_SP(D_ctd_top.sal,556,D_ctd_top.lon,D_ctd_top.lat)
# D_ctd_mid['abs_sal'] = gsw.SA_from_SP(D_ctd_mid.sal,772,D_ctd_mid.lon,D_ctd_mid.lat)

In [ ]:
# TNBD_ctd_top['theta'] = gsw.pt0_from_t(TNBD_ctd_top.abs_sal,TNBD_ctd_top.temp,1000)
# TNBD_ctd_bot['theta'] = gsw.pt0_from_t(TNBD_ctd_bot.abs_sal,TNBD_ctd_bot.temp,1100)

# DITD_ctd_top['theta'] = gsw.pt0_from_t(DITD_ctd_top.abs_sal,DITD_ctd_top.temp,1100)
# DITD_ctd_bot['theta'] = gsw.pt0_from_t(DITD_ctd_bot.abs_sal,DITD_ctd_bot.temp,1200)

# D_ctd_top['theta'] = gsw.pt0_from_t(D_ctd_top.abs_sal,D_ctd_top.temp,556)
# D_ctd_mid['theta'] = gsw.pt0_from_t(D_ctd_mid.abs_sal,D_ctd_mid.temp,772)

In [ ]:
fig = plt.figure()
plt.plot(TNBD_vel_bot.time,TNBD_vel_bot.v)

In [ ]:
# resample datasets to 15 min - this is coarsest sampling frequncy from TNBD current meter 
# TNBD_ctd_15 = TNBD_ctd_bot.resample(time='15min').mean()
# DITD_ctd_15 = DITD_ctd_top.resample(time='15min').mean()
# DITD_vel_15 = DITD_vel_top.resample(time='15min').mean()

In [ ]:
# resample datasets to 30 min - coarsest sampling freq from D current meter 
# TNBD_ctd_30 = TNBD_ctd_bot.resample(time='30min').mean()
# TNBD_vel_30 = TNBD_vel_bot.resample(time='30min').mean()
# DITD_ctd_30 = DITD_ctd_top.resample(time='30min').mean()
# DITD_vel_30 = DITD_vel_top.resample(time='30min').mean()

In [ ]:
# # 03-14 to 01-01
# time_slice = slice('2018-03-14', '2019-01-01')
# TNBD_ctd = TNBD_ctd_15.sel(time=time_slice)
# DITD_ctd = DITD_ctd_15.sel(time=time_slice)
# TNBD_vel = TNBD_vel_bot.sel(time=time_slice)
# DITD_vel = DITD_vel_15.sel(time=time_slice)


In [ ]:
# time_slice = slice('2018-03-14', '2019-01-01')
# TNBD_ctd = TNBD_ctd_30.sel(time=time_slice)
# DITD_ctd = DITD_ctd_30.sel(time=time_slice)
# TNBD_vel = TNBD_vel_30.sel(time=time_slice)
# DITD_vel = DITD_vel_30.sel(time=time_slice)
# D_ctd = D_ctd_mid.sel(time=time_slice)
# D_vel = D_vel_bot.sel(time=time_slice)

In [ ]:
len(TNBD_ctd_top.time), len(DITD_ctd_top.time), len(DITD_vel_top.time), len(TNBD_vel_top.time), len(D_vel_mid.time), len(D_ctd_top.time)

In [ ]:
# D_vel = D_vel.resample(time='30min').mean()
# D_ctd = D_ctd.resample(time='30min').mean()

In [ ]:
# # save intermediate datasets for easy loading later: 

# TNBD_ctd_top.to_netcdf('data/TNBD_ctd_top_30min.nc')
# TNBD_ctd_bot.to_netcdf('data/TNBD_ctd_bot_30min.nc')
# TNBD_vel_top.to_netcdf('data/TNBD_vel_top_30min.nc')
# TNBD_vel_bot.to_netcdf('data/TNBD_vel_bot_30min.nc')

# DITD_ctd_top.to_netcdf('data/DITD_ctd_top_30min.nc')
# DITD_ctd_bot.to_netcdf('data/DITD_ctd_bot_30min.nc')
# DITD_vel_top.to_netcdf('data/DITD_vel_top_30min.nc')
# DITD_vel_bot.to_netcdf('data/DITD_vel_bot_30min.nc')      # sensor cut out in june 2018

# D_ctd_top.to_netcdf('data/D_ctd_top_30min.nc')
# D_ctd_mid.to_netcdf('data/D_ctd_mid_30min.nc')
# # D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# # D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
# D_vel_mid.to_netcdf('data/D_vel_mid_30min.nc')
# D_vel_bot.to_netcdf('data/D_vel_bot_30min.nc')

### Rotate velocity and add to datasets 

In [ ]:
DITD_v_along, DITD_v_across = rotate_velocities(DITD_vel_top.u, DITD_vel_top.v, theta_deg)
DITD_vel_top['u_prime'] = ('time',DITD_v_along.data)
DITD_vel_top['v_prime'] = ('time',DITD_v_across.data)

In [ ]:
DITD_v_along, DITD_v_across = rotate_velocities(DITD_vel_bot.u, DITD_vel_bot.v, theta_deg)
DITD_vel_bot['u_prime'] = ('time',DITD_v_along.data)
DITD_vel_bot['v_prime'] = ('time',DITD_v_across.data)

In [ ]:
TNBD_v_along, TNBD_v_across = rotate_velocities(TNBD_vel_top.u, TNBD_vel_top.v, theta_deg)
TNBD_vel_top['u_prime'] = ('time',TNBD_v_along.data)
TNBD_vel_top['v_prime'] = ('time',TNBD_v_across.data)

In [ ]:
TNBD_v_along, TNBD_v_across = rotate_velocities(TNBD_vel_bot.u, TNBD_vel_bot.v, theta_deg)
TNBD_vel_bot['u_prime'] = ('time',TNBD_v_along.data)
TNBD_vel_bot['v_prime'] = ('time',TNBD_v_across.data)

In [ ]:
D_v_along, D_v_across = rotate_velocities(D_vel_bot.u, D_vel_bot.v, theta_deg)
D_vel_bot['u_prime'] = ('time',D_v_along.data)
D_vel_bot['v_prime'] = ('time',D_v_across.data)

In [ ]:
D_v_along, D_v_across = rotate_velocities(D_vel_mid.u, D_vel_mid.v, theta_deg)
D_vel_mid['u_prime'] = ('time',D_v_along.data)
D_vel_mid['v_prime'] = ('time',D_v_across.data)

In [ ]:
fig = plt.figure()
plt.plot(DITD_vel_top.u,DITD_vel_top.v,label='in-situ')
plt.plot(DITD_vel_top.u_prime,DITD_vel_top.v_prime,label='rotated')
plt.legend()

In [ ]:
fig = plt.figure()
plt.plot(TNBD_vel_bot.u,TNBD_vel_bot.v,label='in-situ')
plt.plot(TNBD_vel_bot.u_prime,TNBD_vel_bot.v_prime,label='rotated')
plt.legend()

In [ ]:
fig = plt.figure()
plt.plot(D_vel_bot.u,D_vel_bot.v,label='in-situ')
plt.plot(D_vel_bot.u_prime,D_vel_bot.v_prime,label='rotated')
plt.legend()

### Investigate relationship between velocity and salinity 

No relationship found 

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, constrained_layout=True, figsize=(10,6))

ax[0,0].scatter(TNBD_vel_bot.u,TNBD_ctd_bot.sal,s=2)
ax[0,0].set_xlim(-0.4,0.4)

ax[0,1].scatter(D_vel_bot.u,D_ctd_mid.sal,s=2)
ax[0,1].set_xlim(-0.4,0.4)

ax[0,2].scatter(DITD_vel_top.u,DITD_ctd_top.sal,s=2)
ax[0,2].set_xlim(-0.4,0.4)

ax[1,0].scatter(TNBD_vel_bot.v,TNBD_ctd_bot.sal,s=2)
ax[1,0].set_xlim(-0.4,0.4)

ax[1,1].scatter(D_vel_bot.v,D_ctd_mid.sal,s=2)
ax[1,1].set_xlim(-0.4,0.4)

ax[1,2].scatter(DITD_vel_top.v,DITD_ctd_top.sal,s=2)
ax[1,2].set_xlim(-0.4,0.4)



In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, constrained_layout=True, figsize=(10,6))

ax[0,0].scatter(TNBD_vel_bot.u_prime,TNBD_ctd_bot.sal,s=2)
ax[0,0].set_xlim(-0.4,0.4)

ax[0,1].scatter(D_vel_bot.u_prime,D_ctd_mid.sal,s=2)
ax[0,1].set_xlim(-0.4,0.4)

ax[0,2].scatter(DITD_vel_top.u_prime,DITD_ctd_top.sal,s=2)
ax[0,2].set_xlim(-0.4,0.4)

ax[1,0].scatter(TNBD_vel_bot.v_prime,TNBD_ctd_bot.sal,s=2)
ax[1,0].set_xlim(-0.4,0.4)

ax[1,1].scatter(D_vel_bot.v_prime,D_ctd_mid.sal,s=2)
ax[1,1].set_xlim(-0.4,0.4)

ax[1,2].scatter(DITD_vel_top.v_prime,DITD_ctd_top.sal,s=2)
ax[1,2].set_xlim(-0.4,0.4)

### Estimate vertical extent of HSSW layer using CTD data 

In [ ]:
# load in CTD data 
palmer = xr.open_dataset('data/Palmer_ctd_DrygalskiBasin.nc')
pipers = xr.open_dataset('data/PIPERS_ctd_DrygalskiBasin.nc')

In [ ]:
def add_dist_from_bottom(ds, depth_var="depth"):
    """
    Add a 'dist_from_bottom' variable to the dataset: distance above
    the seafloor for each sample, computed per-station.

    dist_from_bottom = bottom_depth(station) - depth(station, sample)

    So dist_from_bottom = 0 at the deepest sample of each cast,
    and increases toward the surface.
    """
    depth = ds[depth_var]

    # bottom depth per station = max valid depth in that profile
    bottom_depth = depth.max(dim="sample", skipna=True)

    dist_from_bottom = bottom_depth - depth
    dist_from_bottom.attrs["long_name"] = "Distance above bottom"
    dist_from_bottom.attrs["units"] = "m"

    ds = ds.assign(dist_from_bottom=dist_from_bottom)
    return ds

In [ ]:
pipers = add_dist_from_bottom(pipers)

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).Absolute_Salinity,pipers.isel(station=st).dist_from_bottom,color='k')
# plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.9, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('PIPERS CTD Data')

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).sal,pipers.isel(station=st).depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.72, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Practical Salinity [g/kg]')
plt.title('PIPERS CTD Data')


In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(pipers.station)):
    plt.plot(pipers.isel(station=st).Potential_Temperature,pipers.isel(station=st).depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=-1.85, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Potential Temp')
plt.title('PIPERS CTD Data')

In [ ]:
fig = plt.figure(figsize=(6,8))
for st in range(len(palmer.station)):
    plt.plot(palmer.isel(station=st).Absolute_Salinity,palmer.isel(station=st).Depth,color='k')
plt.gca().invert_yaxis()
# Single vertical line (full height)
plt.axvline(x=34.9, color='r', linestyle='--', linewidth=2, label='x=2')
plt.ylabel('Depth [m]')
plt.xlabel('Absolute Salinity [g/kg]')
plt.title('Palmer CTD Data')
plt.xlim(33.6,35.25)

In [ ]:
def hssw_thickness_1(ds, salinity_var="SA", depth_var="depth", threshold=34.9):
    """
    Compute HSSW layer thickness for each station in a CTD dataset.

    HSSW top boundary = shallowest depth below which ALL samples
    remain above the salinity threshold (i.e. the layer is contiguous
    down to the bottom of the cast). This is more robust to noisy
    spikes near the boundary than a simple "first exceedance" approach.

    Thickness = max profile depth - HSSW top depth

    Returns a DataArray of thickness values (one per station), with NaN
    for stations that never reach the HSSW threshold, or where the
    bottom-most sample itself is below threshold (no HSSW at depth).
    """
    thicknesses = []
    station_ids = []

    for stn in ds["station"].values:
        profile = ds.sel(station=stn)

        depth = profile[depth_var].values
        salt = profile[salinity_var].values

        # drop NaNs
        valid = np.isfinite(depth) & np.isfinite(salt)
        depth = depth[valid]
        salt = salt[valid]

        if depth.size == 0:
            thicknesses.append(np.nan)
            station_ids.append(stn)
            continue

        # sort by depth ascending (shallow -> deep), in case sample
        # order doesn't already correspond to depth order
        order = np.argsort(depth)
        depth = depth[order]
        salt = salt[order]

        max_depth = depth[-1]

        # working from the bottom (deepest) sample upward, find the
        # shallowest index from which every sample to the bottom is
        # above threshold
        above = salt > threshold

        if not above[-1]:
            # bottom of cast isn't even HSSW -> no HSSW layer at depth
            thickness = np.nan
        else:
            # walk upward from the bottom while still above threshold
            idx = len(above) - 1
            while idx > 0 and above[idx - 1]:
                idx -= 1
            hssw_top_depth = depth[idx]
            thickness = max_depth - hssw_top_depth

        thicknesses.append(thickness)
        station_ids.append(stn)

    return xr.DataArray(
        thicknesses,
        coords={"station": station_ids},
        dims="station",
        name="HSSW_thickness"
    )

In [ ]:
thickness_pipers_1 = hssw_thickness_1(pipers,salinity_var='Absolute_Salinity', depth_var='depth',threshold=34.9)
thickness_palmer_1 = hssw_thickness_1(palmer,salinity_var='Absolute_Salinity',depth_var='Depth',threshold=34.9)

In [ ]:
thickness_palmer_1

In [ ]:
thickness_pipers_1.mean()

In [ ]:
thickness_pipers_1.median()

In [ ]:
thickness_pipers_1.min().values, thickness_pipers_1.max().values

In [ ]:
all_thickness = xr.concat([thickness_pipers_1, thickness_palmer_1], dim="station")

mean_thickness = all_thickness.mean(skipna=True)
n_with_hssw = all_thickness.notnull().sum().item()
n_total = all_thickness.size

print(f"Mean HSSW thickness: {mean_thickness.values:.1f} m")
print(f"HSSW present in {n_with_hssw}/{n_total} profiles")

In [ ]:
# plot mooring D to see if HSSW always present at 550 meters 
fig = plt.figure(figsize=(12,4))
plt.plot(D_ctd_top.time,D_ctd_top.abs_sal,label='550m')
plt.plot(D_ctd_mid.time,D_ctd_mid.abs_sal, label='770m')
plt.axhline(y=34.9, color='r', linestyle='--', linewidth=2, label='HSSW Threshold')
plt.legend()

### Estimate Width of HSSW In Trough at Each Mooring 

In [ ]:
from TNB_tools import trough_cross_section

In [ ]:
def extract_cross_section_bedmachine(ds_bathy,
                                      mooring_lon, mooring_lat,
                                      trough_theta_deg,
                                      transect_half_width_deg=0.5,
                                      stop_depth_contour=None,
                                      n_points=1000,
                                      include_ice_cavity=True):
    """
    Extract a bathymetric cross-section from BedMachine Antarctica,
    perpendicular to the trough axis through the mooring location.

    Identical interface to extract_cross_section() but handles:
      - Polar stereographic input grid (EPSG:3031)
      - Sub-ice cavity bathymetry (mask == 3, floating ice)
      - Grounded ice / land masking (mask == 1 or 2 → NaN)

    Parameters
    ----------
    ds_bathy           : xarray Dataset, BedMachine TNB subset
    mooring_lon/lat    : float, mooring position (decimal degrees)
    trough_theta_deg   : float, trough axis angle CCW from East
    include_ice_cavity : bool, if True include sub-ice ocean beneath
                         floating ice (mask==3). If False, treat as land.
                         Default True — use the BedMachine advantage.
    """

    from pyproj import Transformer
    from scipy.interpolate import RegularGridInterpolator
    
    mooring_lon = float(mooring_lon)
    mooring_lat = float(mooring_lat)

    # ── Build transect in lon/lat then convert to PS ──────────────────────────
    perp_theta = np.deg2rad(trough_theta_deg + 90.0)
    lat_scale  = np.cos(np.deg2rad(mooring_lat))

    t             = np.linspace(-transect_half_width_deg,
                                 transect_half_width_deg, n_points)
    transect_lons = mooring_lon + t * np.cos(perp_theta) / lat_scale
    transect_lats = mooring_lat + t * np.sin(perp_theta)

    # Convert transect to polar stereographic
    transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3031',
                                        always_xy=True)
    transect_x, transect_y = transformer.transform(transect_lons,
                                                    transect_lats)

    # ── Interpolate bed and mask onto transect ────────────────────────────────
    x_grid = ds_bathy['x'].values.astype(float)
    y_grid = ds_bathy['y'].values.astype(float)
    bed    = ds_bathy['bed'].values.astype(float)
    mask   = ds_bathy['mask'].values.astype(float)

    # y may be descending — RegularGridInterpolator needs ascending
    if y_grid[0] > y_grid[-1]:
        y_grid = y_grid[::-1]
        bed    = bed[::-1, :]
        mask   = mask[::-1, :]

    interp_bed = RegularGridInterpolator(
        (y_grid, x_grid), bed,
        method='linear', bounds_error=False, fill_value=np.nan
    )
    interp_mask = RegularGridInterpolator(
        (y_grid, x_grid), mask,
        method='nearest', bounds_error=False, fill_value=np.nan
    )

    pts        = np.column_stack([transect_y, transect_x])
    bed_vals   = interp_bed(pts)
    mask_vals  = interp_mask(pts)

    # ── Convert bed elevation to depth (positive down) ────────────────────────
    # bed is negative below sea level in BedMachine
    depth_raw = -bed_vals   # positive down

    # Mask land and grounded ice as NaN
    is_ocean = (mask_vals == 0) | (mask_vals == 3 if include_ice_cavity
                                    else False)
    depth_raw = np.where(is_ocean, depth_raw, np.nan)

    # ── Distance array (flat-Earth, km) ──────────────────────────────────────
    dlons_km = 6371.0 * lat_scale * np.deg2rad(np.diff(transect_lons))
    dlats_km = 6371.0 * np.deg2rad(np.diff(transect_lats))
    dist_km  = np.concatenate([[0], np.cumsum(
        np.sqrt(dlons_km**2 + dlats_km**2))])

    # ── Trim to stop_depth_contour ────────────────────────────────────────────
    if stop_depth_contour is not None:
        mid       = len(depth_raw) // 2
        left_idx  = 0
        right_idx = len(depth_raw) - 1

        for i in range(mid, -1, -1):
            d = depth_raw[i]
            if np.isnan(d) or d < stop_depth_contour:
                left_idx = i; break

        for i in range(mid, len(depth_raw)):
            d = depth_raw[i]
            if np.isnan(d) or d < stop_depth_contour:
                right_idx = i; break

        depth_raw     = depth_raw[left_idx:right_idx + 1]
        mask_vals     = mask_vals[left_idx:right_idx + 1]
        transect_lons = transect_lons[left_idx:right_idx + 1]
        transect_lats = transect_lats[left_idx:right_idx + 1]
        dist_km       = dist_km[left_idx:right_idx + 1]
        dist_km       = dist_km - dist_km[0]

        print(f"Trimmed to {stop_depth_contour}m contour: "
              f"{dist_km[-1]:.1f} km wide")

    # ── Cardinal labels ───────────────────────────────────────────────────────
    from TNB_tools.trough_cross_section import _cardinal_label
    dlon        = transect_lons[-1] - transect_lons[0]
    dlat        = transect_lats[-1] - transect_lats[0]
    left_label  = _cardinal_label(-dlon, -dlat)
    right_label = _cardinal_label( dlon,  dlat)

    # Report ice cavity coverage
    n_cavity = np.sum(mask_vals == 3)
    if n_cavity > 0:
        cavity_km = n_cavity * np.mean(np.diff(dist_km))
        print(f"Sub-ice cavity spans {cavity_km:.1f} km of transect")
        print(f"  → BedMachine resolves seafloor beneath Drygalski Ice Tongue")

    print(f"Transect: {left_label} → {right_label}")

    return (dist_km, depth_raw, transect_lons, transect_lats,
            left_label, right_label, mask_vals)

In [ ]:
# Extract cross-section through DITD
dist1, depth1, t_lons1, t_lats1, left1, right1, mask1 = \
    extract_cross_section_bedmachine(
        ds_bathy             = bath,
        mooring_lon          = DITD_ctd_top.lon.values,
        mooring_lat          = DITD_ctd_top.lat.values,
        trough_theta_deg     = theta_deg,
        transect_half_width_deg = 0.5,
        stop_depth_contour   = 900,
        include_ice_cavity   = True,
    )

# # find_mooring_position_on_transect works unchanged (uses lon/lat)
# from trough_cross_section import (find_mooring_position_on_transect,
#                                    compute_hssw_area)

moor_dist1 = trough_cross_section.find_mooring_position_on_transect(
    dist1, t_lons1, t_lats1, DITD_ctd_top.lon, DITD_ctd_top.lat)

# compute_hssw_area works unchanged — just pass the new depth array
# NaN values (land/grounded ice) are handled by np.maximum(..., 0)
area1, width1, results1 = trough_cross_section.compute_hssw_area(
    dist1, depth1,
    hssw_thickness_m = 500.0,
    mooring_dist_km  = moor_dist1,
    left_label       = left1,
    right_label      = right1,
    west_left        = True,
)

In [ ]:
# ── M2 cross-section (repeat for second along-trough mooring) ─────────────
dist2, depth2, t_lons2, t_lats2, left2, right2, mask2 = extract_cross_section_bedmachine(
    ds_bathy                = bath,
    mooring_lon             = D_vel_bot.lon,
    mooring_lat             = D_vel_bot.lat,
    trough_theta_deg        = theta_deg,
    transect_half_width_deg = 0.5,
    stop_depth_contour      = 900,
    include_ice_cavity   = True,   
)

moor_dist2 = trough_cross_section.find_mooring_position_on_transect(
    dist2, t_lons2, t_lats2, D_vel_bot.lon, D_vel_bot.lat
)

area2, width2, results2 = trough_cross_section.compute_hssw_area(
    dist2, depth2,
    hssw_thickness_m = 500.0,
    mooring_dist_km  = moor_dist2,
    left_label       = left2,
    right_label      = right2,
    west_left        = True,
)

In [ ]:
# ── M3 cross-section (repeat for second along-trough mooring) ─────────────
dist3, depth3, t_lons3, t_lats3, left3, right3, mask3 = extract_cross_section_bedmachine(
    ds_bathy                = bath,
    mooring_lon             = TNBD_vel_top.lon,
    mooring_lat             = TNBD_vel_top.lat,
    trough_theta_deg        = theta_deg,
    transect_half_width_deg = 0.5,
    stop_depth_contour      = 900,
    include_ice_cavity   = True,
)

moor_dist3 = trough_cross_section.find_mooring_position_on_transect(
    dist3, t_lons3, t_lats3, TNBD_vel_top.lon, TNBD_vel_top.lat
)

area3, width3, results3 = trough_cross_section.compute_hssw_area(
    dist3, depth3,
    hssw_thickness_m = 500.0,
    mooring_dist_km  = moor_dist3,
    left_label       = left3,
    right_label      = right3,
    west_left        = True,
)

In [ ]:
def _add_transect_endticks(ax, lons, lats, color='#D85A30', ticklen=0.015):
    """
    Perpendicular tick marks at both ends of a transect line.
    Standard cartographic convention indicating a finite cross-section face.
    """
    dlon = lons[-1] - lons[0]
    dlat = lats[-1] - lats[0]
    norm = np.sqrt(dlon**2 + dlat**2)
    tx, ty = dlon / norm, dlat / norm
    px, py = -ty, tx
    for lon_end, lat_end in [(lons[0], lats[0]), (lons[-1], lats[-1])]:
        ax.plot([lon_end - px * ticklen, lon_end + px * ticklen],
                [lat_end - py * ticklen, lat_end + py * ticklen],
                color=color, lw=1.8, zorder=5)


def _add_north_arrow(ax, x=0.96, y=0.18, length=0.07):
    """North arrow in axis coordinates."""
    ax.annotate('', xy=(x, y + length), xytext=(x, y),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))
    ax.text(x, y - 0.03, 'N', transform=ax.transAxes,
            ha='center', va='top', fontsize=9,
            color='white', fontweight='bold')


def _add_scale_bar(ax, ref_lat, x0_frac=0.65, y0_frac=0.05,
                   target_km=20):
    """
    Scale bar converting km to degrees longitude at ref_lat.
    target_km sets the approximate bar length in km.
    """
    R             = 6371.0
    km_per_deg    = R * np.cos(np.deg2rad(ref_lat)) * np.pi / 180.0
    bar_deg       = target_km / km_per_deg
    xlim, ylim    = ax.get_xlim(), ax.get_ylim()
    x0 = xlim[0] + x0_frac * (xlim[1] - xlim[0])
    y0 = ylim[0] + y0_frac * (ylim[1] - ylim[0])
    ax.plot([x0, x0 + bar_deg], [y0, y0],
            color='white', lw=2.5, solid_capstyle='butt', zorder=5)
    for xpt in [x0, x0 + bar_deg]:
        ax.plot([xpt, xpt], [y0 - 0.005, y0 + 0.005],
                color='white', lw=1.5, zorder=5)
    ax.text(x0 + bar_deg / 2, y0 + 0.012, f'{target_km} km',
            ha='center', va='bottom', fontsize=8,
            color='white', fontweight='bold', zorder=5)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from pyproj import Transformer


def plot_bathy_with_transect_bedmachine(ds_bathy,
                                         moorings=None,
                                         lon_bounds=None, lat_bounds=None,
                                         depth_contours=(200, 500, 800, 900, 1000, 1100, 1200),
                                         contour_label_every=2,
                                         save_path=None):
    """
    Plan-view bathymetry map using BedMachine Antarctica data, styled to
    match the study-site figure (Blues bathymetry, grey land, translucent
    floating ice with red outline, white isobaths, white geographic labels).

    Parameters
    ----------
    ds_bathy : xarray.Dataset
        BedMachine dataset with 'x', 'y', 'bed', 'mask'.
    moorings : list of dict
        Each dict describes one mooring + its transect, e.g.:
        {
            'name':  'M1',
            'lon':   float,
            'lat':   float,
            'color': 'tomato',                  # optional, auto-assigned if omitted
            'transect_lons': array-like,        # optional
            'transect_lats': array-like,        # optional
        }
        Any number of moorings supported.
    lon_bounds, lat_bounds : tuple, optional
        Map extent. Auto-computed from transects (with padding) if omitted.
    depth_contours : sequence of int/float
        Exact isobath levels to draw, in meters. Pass any set you like,
        e.g. (300, 600, 1000) or (200, 400, 600, 800, 1000, 1200, 1400).
    contour_label_every : int
        Label every Nth contour level (to avoid crowding). Set to 1 to
        label all, or a large number to label none practically.
    save_path : str
        Default save path if you choose to save the figure.

    Returns
    -------
    fig, ax
    """
    if moorings is None:
        moorings = []

    plt.rcParams.update({
        'font.size': 12,
        'axes.labelsize': 12,
        'axes.titlesize': 12,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'legend.title_fontsize': 12,
    })

    default_colors = ['tomato', 'seagreen', 'steelblue', 'gold',
                       'mediumorchid', 'darkorange', 'slategray', 'crimson']
    for i, m in enumerate(moorings):
        if 'color' not in m or m['color'] is None:
            m['color'] = default_colors[i % len(default_colors)]

    # ── Convert BedMachine PS grid to lon/lat ─────────────────────────────────
    transformer_inv = Transformer.from_crs('EPSG:3031', 'EPSG:4326',
                                            always_xy=True)
    x_2d, y_2d     = np.meshgrid(ds_bathy['x'].values, ds_bathy['y'].values)
    lon_2d, lat_2d = transformer_inv.transform(x_2d, y_2d)

    bed  = ds_bathy['bed'].values.astype(float)
    mask = ds_bathy['mask'].values.astype(float)

    depth      = np.where(bed < 0, -bed, np.nan)
    ocean_mask = (mask == 0) | (mask == 3)
    ocean_depth = np.where(ocean_mask, depth, np.nan)
    land        = np.where((mask == 1) | (mask == 2), 1.0, np.nan)
    floating    = np.where(mask == 3, 1.0, np.nan)

    # ── Auto bounds from transects/moorings ───────────────────────────────────
    if lon_bounds is None or lat_bounds is None:
        all_lons, all_lats = [], []
        for m in moorings:
            all_lons.append(m['lon'])
            all_lats.append(m['lat'])
            if m.get('transect_lons') is not None:
                all_lons.extend(np.atleast_1d(m['transect_lons']))
                all_lats.extend(np.atleast_1d(m['transect_lats']))
        if not all_lons:
            raise ValueError("No moorings/transects given — must supply "
                              "lon_bounds and lat_bounds explicitly.")
        all_lons, all_lats = np.array(all_lons), np.array(all_lats)
        pad = 0.3
        if lon_bounds is None:
            lon_bounds = (all_lons.min() - pad, all_lons.max() + pad)
        if lat_bounds is None:
            lat_bounds = (all_lats.min() - pad, all_lats.max() + pad)

    # ── Mask to plot bounds ────────────────────────────────────────────────────
    in_bounds  = ((lon_2d >= lon_bounds[0]) & (lon_2d <= lon_bounds[1]) &
                  (lat_2d >= lat_bounds[0]) & (lat_2d <= lat_bounds[1]))
    ocean_plot = np.where(in_bounds, ocean_depth, np.nan)

    depth_ocean = ocean_plot[np.isfinite(ocean_plot)]
    if depth_ocean.size == 0:
        raise ValueError("No ocean values in bounds — check lon/lat bounds.")
    vmax = np.nanpercentile(depth_ocean, 98)

    # ── Figure ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True )

    # Bathymetry
    pcm = ax.pcolormesh(lon_2d, lat_2d, ocean_plot,
                         cmap='Blues', shading='auto',
                         vmin=0, vmax=vmax, zorder=1)
    cbar = plt.colorbar(pcm, ax=ax, fraction=0.025, aspect=30)
    cbar.set_label('Depth (m)', fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    # Land / grounded ice
    ax.pcolormesh(lon_2d, lat_2d, land,
                  cmap='Greys', shading='auto',
                  vmin=0, vmax=1, alpha=0.85, zorder=2)

    # Floating ice — translucent white + red outline
    ax.pcolormesh(lon_2d, lat_2d, floating,
                  cmap='Greys_r', shading='auto',
                  vmin=0, vmax=1, alpha=0.3, zorder=3)
    ax.contour(lon_2d, lat_2d, (mask == 3).astype(float),
               levels=[0.5], colors=['tab:red'], linewidths=1.4,
               linestyles='-', zorder=5)

    # ── Isobaths — exact user-specified levels ─────────────────────────────────
    valid_contours = [c for c in depth_contours
                       if np.nanmin(ocean_plot[np.isfinite(ocean_plot)]) < c < vmax]
    if valid_contours:
        cs = ax.contour(lon_2d, lat_2d, ocean_plot,
                         levels=valid_contours,
                         colors='white', linewidths=0.6,
                         alpha=0.5, zorder=6)
        labels_to_show = valid_contours[::contour_label_every] if contour_label_every > 0 else []
        if labels_to_show:
            ax.clabel(cs, levels=labels_to_show, fmt='%d m',
                      fontsize=7, colors='white', inline=True,
                      inline_spacing=4)

    # ── Transects ────────────────────────────────────────────────────────────
    for m in moorings:
        if m.get('transect_lons') is not None:
            ax.plot(m['transect_lons'], m['transect_lats'],
                    color=m['color'], lw=2.0, zorder=4,
                    label=f"Transect {m['name']}")
            _add_transect_endticks(ax, m['transect_lons'], m['transect_lats'],
                                    color=m['color'])

    # ── Moorings — styled like study-site figure (circle markers, white edge) ──
    for m in moorings:
        ax.plot(m['lon'], m['lat'],
                marker='o', markersize=10, color=m['color'],
                markeredgecolor='white', markeredgewidth=1.5,
                linestyle='none', zorder=9)
        txt = ax.text(m['lon'], m['lat'] + 0.05, m['name'],
                      ha='center', va='bottom', fontsize=11,
                      fontweight='bold', color=m['color'], zorder=9)
        txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground='white')])

    # ── North arrow / scale bar ─────────────────────────────────────────────
    _add_north_arrow(ax)
    _add_scale_bar(ax, np.mean(lat_bounds))

    # ── Axes formatting ─────────────────────────────────────────────────────
    ax.set_xlim(lon_bounds)
    ax.set_ylim(lat_bounds)
    lat_mean = np.mean(lat_bounds)
    ax.set_aspect(1.0 / np.cos(np.deg2rad(lat_mean)))
    ax.set_xlabel('Longitude (°E)', fontsize=12, labelpad=6)
    ax.set_ylabel('Latitude (°N)', fontsize=12, labelpad=6)
    # ax.set_title('Drygalski Trough — BedMachine bathymetry and transects',
    #              fontsize=12, loc='left', fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}°'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:.2f}°'))
    ax.tick_params(labelsize=10)

    if moorings:
        leg = ax.legend(loc='upper left', fontsize=9,
                        framealpha=0.75, edgecolor='#aaaaaa',
                        facecolor='#1a2a3a', labelcolor='white',
                        title='Legend', title_fontsize=10)
        leg.get_title().set_color('white')

    # plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    return fig, ax


def _add_transect_endticks(ax, lons, lats, color):
    """Small perpendicular ticks at transect endpoints."""
    ax.plot(lons[0], lats[0], marker='|', markersize=8,
            color=color, zorder=5)
    ax.plot(lons[-1], lats[-1], marker='|', markersize=8,
            color=color, zorder=5)


def _add_north_arrow(ax):
    ax.annotate('', xy=(0.96, 0.20), xytext=(0.96, 0.13),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='white',
                                lw=2.0, mutation_scale=15))
    ax.text(0.96, 0.10, 'N', transform=ax.transAxes,
            ha='center', va='top', fontsize=10,
            color='white', fontweight='bold')


def _add_scale_bar(ax, lat_mean, length_km=25):
    R = 6371.0
    km_per_deg = R * np.cos(np.deg2rad(lat_mean)) * np.pi / 180.0
    bar_deg = length_km / km_per_deg
    xlim, ylim = ax.get_xlim(), ax.get_ylim()

    sb_x = xlim[1] - 0.15 * (xlim[1] - xlim[0])
    sb_y = ylim[1] - 0.03 * (ylim[1] - ylim[0])

    ax.plot([sb_x, sb_x + bar_deg], [sb_y, sb_y],
            color='white', lw=3, solid_capstyle='butt', zorder=12)
    for xpt in [sb_x, sb_x + bar_deg]:
        ax.plot([xpt, xpt], [sb_y - 0.004, sb_y + 0.004],
                color='white', lw=2, zorder=12)
    ax.text(sb_x, sb_y - 0.012, '0',
            ha='center', va='top', fontsize=8,
            color='white', fontweight='bold', zorder=12)
    ax.text(sb_x + bar_deg, sb_y - 0.012, f'{length_km} km',
            ha='center', va='top', fontsize=8,
            color='white', fontweight='bold', zorder=12)


# ── EXAMPLE USAGE ─────────────────────────────────────────────────────────────
# fig, ax = plot_bathy_with_transect_bedmachine(
#     ds_bathy = ds_TNB,
#     moorings = [
#         {'name': 'DITD', 'lon': DITD_ctd_top.lon, 'lat': DITD_ctd_top.lat,
#          'color': 'tomato', 'transect_lons': t_lons1, 'transect_lats': t_lats1},
#         {'name': 'TNBD', 'lon': TNBD_ctd_bot.lon, 'lat': TNBD_ctd_bot.lat,
#          'color': 'steelblue', 'transect_lons': t_lons2, 'transect_lats': t_lats2},
#         {'name': 'D', 'lon': D_vel.lon, 'lat': D_vel.lat,
#          'color': 'seagreen', 'transect_lons': t_lons3, 'transect_lats': t_lats3},
#     ],
#     lon_bounds      = (163.0, 165.5),
#     lat_bounds      = (-76.0, -74.5),
#     depth_contours  = (300, 600, 900, 1200),   # ← any levels you want, any count
#     contour_label_every = 1,
# )
# plt.show()

In [ ]:
# ── EXAMPLE USAGE ─────────────────────────────────────────────────────────────
fig, ax = plot_bathy_with_transect_bedmachine(
    ds_bathy = bath,
    moorings = [
        {'name': 'DITD', 'lon': DITD_ctd_top.lon, 'lat': DITD_ctd_top.lat,
         'color': 'tomato', 'transect_lons': t_lons1, 'transect_lats': t_lats1},
        {'name': 'TNBD', 'lon': TNBD_ctd_bot.lon, 'lat': TNBD_ctd_bot.lat,
         'color': 'steelblue', 'transect_lons': t_lons3, 'transect_lats': t_lats3},
        {'name': 'D', 'lon': D_vel_bot.lon, 'lat': D_vel_bot.lat,
         'color': 'seagreen', 'transect_lons': t_lons2, 'transect_lats': t_lats2},
    ],
    lon_bounds      = (161.5, 167),
    lat_bounds      = (-76.0, -74.5),
    depth_contours  = (300, 600, 900, 1200),   # ← any levels you want, any count
    contour_label_every = 1,
    save_path='mooring_cross_section_map.png'
)
plt.show()

### Low pass filter (3 or 2 day cosine window) to remove tides 

In [ ]:
from TNB_tools.tides.tidal_decomposition import decompose_tidal_dataset

In [ ]:
# load in functions 
D_vel_lowpass, D_vel_highpass = decompose_tidal_dataset(
    D_vel_bot, var_names=["u", "v","u_prime", "v_prime"], window_days=2, sampling_hours=0.5
)

In [ ]:
TNBD_vel_lowpass, TNBD_vel_highpass = decompose_tidal_dataset(
    TNBD_vel_bot, var_names=["u", "v","u_prime", "v_prime"], window_days=2, sampling_hours=0.5
)

DITD_vel_lowpass, DITD_vel_highpass = decompose_tidal_dataset(
    DITD_vel_top, var_names=["u", "v","u_prime", "v_prime"], window_days=2, sampling_hours=0.5
)

### Compute Volume Transports

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy.signal import butter, filtfilt

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
RHO   = 1028.0   # kg/m³
CP    = 3985.0   # J/(kg·K)
T_REF = -1.9     # °C, surface freezing point reference


# ══════════════════════════════════════════════════════════════════════════════
# TRANSPORT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def compute_volume_transport(v_along, area_m2):
    """
    Volume transport time series.

    Parameters
    ----------
    v_along  : np.ndarray, along-trough velocity (m/s)
    area_m2  : float, cross-sectional area (m²)

    Returns
    -------
    Q_m3s : np.ndarray, volume transport (m³/s)
    Q_Sv  : np.ndarray, volume transport (Sv, 1 Sv = 1e6 m³/s)
    """
    Q_m3s = v_along * area_m2
    Q_Sv  = Q_m3s / 1e6
    return Q_m3s, Q_Sv


def compute_salt_transport(v_along, salinity, area_m2, rho=RHO):
    """
    Salt transport time series.

    Parameters
    ----------
    v_along  : np.ndarray, along-trough velocity (m/s)
    salinity : np.ndarray, absolute salinity (g/kg)
    area_m2  : float, cross-sectional area (m²)
    rho      : float, reference density (kg/m³)

    Returns
    -------
    F_salt : np.ndarray, salt transport (g/s)
    """
    F_salt = rho * v_along * salinity * area_m2
    return F_salt


def compute_heat_transport(v_along, temperature, area_m2,
                            rho=RHO, cp=CP, T_ref=T_REF):
    """
    Heat transport time series relative to T_ref.

    Parameters
    ----------
    v_along     : np.ndarray, along-trough velocity (m/s)
    temperature : np.ndarray, in-situ temperature (°C)
    area_m2     : float, cross-sectional area (m²)
    rho         : float, reference density (kg/m³)
    cp          : float, specific heat capacity (J/kg/K)
    T_ref       : float, reference temperature (°C), default -1.9°C

    Returns
    -------
    F_heat_W  : np.ndarray, heat transport (W)
    F_heat_GW : np.ndarray, heat transport (GW)
    """
    F_heat_W  = rho * cp * v_along * (temperature - T_ref) * area_m2
    F_heat_GW = F_heat_W / 1e9
    return F_heat_W, F_heat_GW


def build_transport_dataset(time, v_along, salinity, temperature,
                             area_m2, mooring_name,
                             dt_hours=1.0,
                             rho=RHO, cp=CP, T_ref=T_REF):
    """
    Compute all three transport time series and store in an xarray Dataset.

    Parameters
    ----------
    time         : array-like, timestamps
    v_along      : np.ndarray, along-trough velocity (m/s)
    salinity     : np.ndarray, salinity (psu)
    temperature  : np.ndarray, temperature (°C)
    area_m2      : float, cross-sectional area (m²)
    mooring_name : str, e.g. 'DITD', 'TNBD'
    dt_hours     : float, sampling interval (hours)

    Returns
    -------
    ds : xarray Dataset with all transport variables and metadata
    """
    Q_m3s, Q_Sv      = compute_volume_transport(v_along, area_m2)
    F_salt            = compute_salt_transport(v_along, salinity,
                                               area_m2, rho)
    F_heat_W, F_heat_GW = compute_heat_transport(v_along, temperature,
                                                   area_m2, rho, cp, T_ref)

    ds = xr.Dataset(
        {
            'v_along':    (['time'], v_along,
                           {'units': 'm/s',
                            'long_name': 'Along-trough velocity'}),
            'salinity':   (['time'], salinity,
                           {'units': 'psu',
                            'long_name': 'Salinity'}),
            'temperature':(['time'], temperature,
                           {'units': 'degC',
                            'long_name': 'Temperature'}),
            'Q_m3s':      (['time'], Q_m3s,
                           {'units': 'm3/s',
                            'long_name': 'Volume transport'}),
            'Q_Sv':       (['time'], Q_Sv,
                           {'units': 'Sv',
                            'long_name': 'Volume transport'}),
            'F_salt':     (['time'], F_salt,
                           {'units': 'kg/s',
                            'long_name': 'Salt transport'}),
            'F_heat_W':   (['time'], F_heat_W,
                           {'units': 'W',
                            'long_name': 'Heat transport'}),
            'F_heat_GW':  (['time'], F_heat_GW,
                           {'units': 'GW',
                            'long_name': 'Heat transport'}),
        },
        coords={'time': time},
        attrs={
            'mooring':    mooring_name,
            'area_m2':    area_m2,
            'T_ref_degC': T_ref,
            'rho_kgm3':   rho,
            'cp_JkgK':    cp,
            'dt_hours':   dt_hours,
        }
    )

    print(f"[{mooring_name}] Transport dataset built: {len(time)} timesteps")
    print(f"  Area:          {area_m2:.3e} m²")
    print(f"  Mean Q:        {float(np.nanmean(Q_Sv)):.4f} Sv")
    print(f"  Mean F_salt:   {float(np.nanmean(F_salt)):.3e} kg/s")
    print(f"  Mean F_heat:   {float(np.nanmean(F_heat_GW)):.4f} GW")

    return ds


# ══════════════════════════════════════════════════════════════════════════════
# STATISTICS FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def integral_timescale(x, dt_hours=0.5, max_lag_hours=240):
    """
    One-sided integral timescale (Saunders 1987).
    Integrates the autocorrelation from lag=0 to first zero crossing.

    Parameters
    ----------
    x           : np.ndarray, time series (NaNs handled)
    dt_hours    : float, sampling interval (hours)
    max_lag_hours: float, maximum lag to search (hours)

    Returns
    -------
    T_L_hours : float, integral timescale (hours)
    ac        : np.ndarray, autocorrelation function
    lags      : np.ndarray, lag values (hours)
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 10:
        return np.nan, np.array([]), np.array([])

    x  = x - np.nanmean(x)
    std = np.nanstd(x)
    if std == 0:
        return 0.0, np.array([1.0]), np.array([0.0])

    max_lag_pts = min(int(max_lag_hours / dt_hours), len(x) // 2)
    lags_pts    = np.arange(max_lag_pts)
    ac = np.array([
        np.corrcoef(x[:-k], x[k:])[0, 1] if k > 0 else 1.0
        for k in lags_pts
    ])

    # Integrate to first zero crossing
    zero_cross = np.where(ac <= 0)[0]
    cutoff     = zero_cross[0] if len(zero_cross) > 0 else len(ac)
    T_L_hours  = float(np.trapezoid(ac[:cutoff]) * dt_hours)
    T_L_hours  = max(T_L_hours, dt_hours)   # floor at sampling interval

    lags_hours = lags_pts * dt_hours
    return T_L_hours, ac, lags_hours


def standard_error(x, T_L_hours, dt_hours=0.5):
    """
    Standard error of the mean accounting for autocorrelation
    (Saunders 1987).

    SE = std(x) / sqrt(N_eff)
    N_eff = N * dt / (2 * T_L)

    Parameters
    ----------
    x         : np.ndarray, time series
    T_L_hours : float, integral timescale from integral_timescale()
    dt_hours  : float, sampling interval (hours)

    Returns
    -------
    SE    : float, standard error
    N_eff : float, effective degrees of freedom
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    N     = len(x)
    std   = np.nanstd(x, ddof=1)
    N_eff = max((N * dt_hours) / (2.0 * T_L_hours), 2.0)
    SE    = std / np.sqrt(N_eff)
    return SE, N_eff


def resample_with_stats(ds, rule, dt_hours=0.5, variables=None,
                         T_L_hours=None):
    """
    If T_L_hours is provided, use it for all windows (full-record T_L).
    If None, compute T_L fresh per window (monthly T_L — current behaviour).
    """
    if variables is None:
        variables = ['Q_Sv', 'F_salt', 'F_heat_GW']

    data_vars = {}

    for var in variables:
        x_full = ds[var].values
        time   = ds['time'].values

        df_tmp = pd.DataFrame({'x': x_full},
                               index=pd.DatetimeIndex(time))
        groups = df_tmp.resample(rule)

        times, means, stds, ses, neffs = [], [], [], [], []

        for t_label, group in groups:
            x_grp = group['x'].values
            x_grp = x_grp[np.isfinite(x_grp)]
            if len(x_grp) < 3:
                continue

            mean = np.nanmean(x_grp)
            std  = np.nanstd(x_grp, ddof=1)

            # ── Key change: use fixed T_L if provided ─────────────────────────
            if T_L_hours is not None:
                # Full-record T_L — conservative, stable
                T_L = T_L_hours
            else:
                # Monthly T_L — computed fresh per window (original behaviour)
                T_L, _, _ = integral_timescale(x_grp, dt_hours=dt_hours)
                if np.isnan(T_L):
                    T_L = dt_hours

            se, n_eff = standard_error(x_grp, T_L, dt_hours)

            times.append(t_label)
            means.append(mean)
            stds.append(std)
            ses.append(se)
            neffs.append(n_eff)

        times = np.array(times, dtype='datetime64[ns]')
        data_vars[f'{var}_mean'] = (['time'], np.array(means),
                                     {'units': ds[var].attrs.get('units', ''),
                                      'long_name': f'{var} mean'})
        data_vars[f'{var}_std']  = (['time'], np.array(stds),
                                     {'units': ds[var].attrs.get('units', ''),
                                      'long_name': f'{var} std dev'})
        data_vars[f'{var}_se']   = (['time'], np.array(ses),
                                     {'units': ds[var].attrs.get('units', ''),
                                      'long_name': f'{var} standard error'})
        data_vars[f'{var}_neff'] = (['time'], np.array(neffs),
                                     {'long_name': f'{var} effective DOF'})

    ds_out = xr.Dataset(
        data_vars,
        coords={'time': times},
        attrs={**ds.attrs,
               'resample_rule':   rule,
               'T_L_method':      'full_record' if T_L_hours is not None
                                  else 'monthly',
               'T_L_hours_used':  T_L_hours if T_L_hours is not None
                                  else 'per_window'}
    )
    return ds_out


# ══════════════════════════════════════════════════════════════════════════════
# EXAMPLE USAGE
# ══════════════════════════════════════════════════════════════════════════════
#
# ── Build transport dataset for each mooring ──────────────────────────────────
#
# ds_DITD = build_transport_dataset(
#     time         = DITD_ctd_top.time.values,
#     v_along      = v_along_DITD,        # low-pass filtered, rotated
#     salinity     = DITD_ctd_top.salinity.values,
#     temperature  = DITD_ctd_top.temperature.values,
#     area_m2      = area_DITD,            # from compute_hssw_area()
#     mooring_name = 'DITD',
#     dt_hours     = 1.0,
# )
#
# ds_TNBD = build_transport_dataset(
#     time         = TNBD_ctd_bot.time.values,
#     v_along      = v_along_TNBD,
#     salinity     = TNBD_ctd_bot.salinity.values,
#     temperature  = TNBD_ctd_bot.temperature.values,
#     area_m2      = area_TNBD,
#     mooring_name = 'TNBD',
#     dt_hours     = 0.5,                  # 30 min sampling
# )
#
# ── Resample to whatever resolution you need for plotting ─────────────────────
#
# # Monthly
# ds_DITD_monthly = resample_with_stats(ds_DITD, rule='ME',  dt_hours=1.0)
# ds_TNBD_monthly = resample_with_stats(ds_TNBD, rule='ME',  dt_hours=0.5)
#
# # Weekly
# ds_DITD_weekly  = resample_with_stats(ds_DITD, rule='W',   dt_hours=1.0)
#
# # 3-day
# ds_DITD_3day    = resample_with_stats(ds_DITD, rule='3D',  dt_hours=1.0)
#
# ── Access variables for plotting ─────────────────────────────────────────────
#
# t     = ds_DITD_monthly.time.values
# Q     = ds_DITD_monthly['Q_Sv_mean'].values
# Q_std = ds_DITD_monthly['Q_Sv_std'].values   # variability shading
# Q_se  = ds_DITD_monthly['Q_Sv_se'].values    # uncertainty error bars
#
# ax.fill_between(t, Q - Q_std, Q + Q_std, alpha=0.2, color='tomato')
# ax.plot(t, Q, color='tomato', lw=1.8)
# ax.errorbar(t, Q, yerr=Q_se, fmt='none', ecolor='tomato',
#             capsize=3, elinewidth=1.2)
#
# ── Access raw transport values at full resolution ────────────────────────────
#
# ds_DITD['Q_Sv'].values           # full resolution time series
# ds_DITD['F_salt'].values
# ds_DITD['F_heat_GW'].values
#
# ── Save to netcdf for later ──────────────────────────────────────────────────
#
# ds_DITD.to_netcdf('transport_DITD.nc')
# ds_TNBD.to_netcdf('transport_TNBD.nc')

In [ ]:
# km^2 
# CROSS SECTIONAL AREA BASED ON TROUGH WIDTH TO 900 M BATHYMETRIC CONTOUR   
A_TNBD = 8.71 * 1e6
A_D = 10.09 * 1e6
A_DITD = 10.39 * 1e6

In [ ]:
# AREA BASED ON ROSSBY RADIUS OF DEFORMATION
A = 500 * (10 * 1e3)

In [ ]:
#── Build transport dataset for each mooring ──────────────────────────────────

ds_DITD = build_transport_dataset(
    time         = DITD_ctd_top.time.values,
    v_along      = DITD_vel_top.u_prime.data,        # low-pass filtered, rotated
    salinity     = DITD_ctd_top.abs_sal.values,
    temperature  = DITD_ctd_top.theta.values,
    area_m2      = A,            # from compute_hssw_area()
    mooring_name = 'DITD',
    dt_hours     = 0.5,
)

ds_TNBD = build_transport_dataset(
    time         = TNBD_ctd_bot.time.values,
    v_along      = TNBD_vel_bot.u_prime.data,
    salinity     = TNBD_ctd_bot.abs_sal.values,
    temperature  = TNBD_ctd_bot.theta.values,
    area_m2      = A,
    mooring_name = 'TNBD',
    dt_hours     = 0.5,                  # 30 min sampling
)

ds_D = build_transport_dataset(
    time         = D_ctd_mid.time.values,
    v_along      = D_vel_bot.u_prime.data,
    salinity     = D_ctd_mid.abs_sal.values,
    temperature  = D_ctd_mid.theta.values,
    area_m2      = A,
    mooring_name = 'D',
    dt_hours     = 0.5,                  # 30 min sampling
)

In [ ]:
# define integral timescales for whole record from previous notebook 
# DAYS 
TNBD_TL = 3.37
DITD_TL = 1.69 
D_TL = 10.70

In [ ]:
fig, ax = plt.subplots(nrows=3,ncols=1,figsize=(12,10),constrained_layout=True)
ax[0].plot(ds_TNBD.time,ds_TNBD.Q_Sv)
ax[1].plot(ds_TNBD.time,ds_TNBD.F_salt)
ax[2].plot(ds_TNBD.time,ds_TNBD.F_heat_W)

In [ ]:
# Monthly
ds_DITD_monthly = resample_with_stats(ds_DITD, rule='MS',  dt_hours=0.5, T_L_hours=DITD_TL*24)
ds_TNBD_monthly = resample_with_stats(ds_TNBD, rule='MS',  dt_hours=0.5, T_L_hours=TNBD_TL*24)
ds_D_monthly = resample_with_stats(ds_D, rule='MS',  dt_hours=0.5,T_L_hours=D_TL*24)

In [ ]:
# Weekly 
ds_DITD_weekly = resample_with_stats(ds_DITD, rule='W',  dt_hours=0.5, T_L_hours=DITD_TL*24)
ds_TNBD_weekly = resample_with_stats(ds_TNBD, rule='W',  dt_hours=0.5, T_L_hours=TNBD_TL*24)
ds_D_weekly = resample_with_stats(ds_D, rule='W',  dt_hours=0.5,T_L_hours=D_TL*24)

In [ ]:
np.round(ds_TNBD_monthly.F_salt_mean.values / 1e9,2)

In [ ]:
np.round(ds_TNBD_monthly.F_heat_GW_mean.values,2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── SETTINGS ──────────────────────────────────────────────────────────────────
COLORS = {
    'TNBD': 'steelblue',
    'D':    'seagreen',
    'DITD': 'tomato',
}

FONT = {
    'title':  13,
    'label':  12,
    'tick':   11,
    'legend': 10,
}

# ── DATA ──────────────────────────────────────────────────────────────────────
datasets = {
    'TNBD': ds_TNBD_monthly,
    'D':    ds_D_monthly,
    'DITD': ds_DITD_monthly,
}

# datasets = {
#     'TNBD': ds_TNBD_weekly,
#     'D':    ds_D_weekly,
#     'DITD': ds_DITD_weekly,
# }

# ── COMPUTE SHARED Y-AXIS RANGE ACROSS ALL THREE MOORINGS ────────────────────
# y_max = 0.0
# for ds in datasets.values():
#     Q   = ds['Q_Sv_mean'].values
#     std = ds['Q_Sv_std'].values
#     y_max = max(y_max, np.nanmax(np.abs(Q) + std) * 1.3)

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    3, 1, figsize=(9, 9),
    sharex=True, sharey=True,
    gridspec_kw={'hspace': 0.08}
)

for ax, (name, ds) in zip(axes, datasets.items()):
    color = COLORS[name]
    t     = ds['time'].values
    Q     = ds['Q_Sv_mean'].values
    std   = ds['Q_Sv_std'].values
    se    = ds['Q_Sv_se'].values

    # ±1 std shading (ocean variability)
    ax.fill_between(t, Q - std, Q + std,
                    color=color, alpha=0.18, zorder=1,
                    label='±1 std')

    # Monthly mean line
    ax.plot(t, Q,
            color=color, lw=2.0, zorder=3,
            label='Monthly mean')

    # ±1 SE error bars (uncertainty on the mean)
    ax.errorbar(t, Q, yerr=se,
                fmt='none', ecolor=color,
                elinewidth=1.3, capsize=4, capthick=1.3,
                zorder=4, label='±1 SE')

    # Zero line
    ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.4, zorder=2)

    # Mooring label
    ax.text(0.01, 0.93, name,
            transform=ax.transAxes,
            fontsize=FONT['title'], fontweight='bold',
            color=color, va='top', ha='left')

    ax.set_ylabel('Volume transport (Sv)', fontsize=FONT['label'])
    ax.tick_params(axis='both', labelsize=FONT['tick'])

    # Legend on top panel only
    if name == 'TNBD':
        ax.legend(fontsize=FONT['legend'], loc='upper right',
                  framealpha=0.8, edgecolor='#cccccc')

# ── Shared y-axis range ───────────────────────────────────────────────────────
axes[0].set_ylim(-1,1)

# ── x-axis formatting ─────────────────────────────────────────────────────────
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].tick_params(axis='x', rotation=30, labelsize=FONT['tick'])

# ── Title ─────────────────────────────────────────────────────────────────────
# fig.suptitle('Monthly HSSW Volume Transport — Drygalski Trough (2018)',
#              fontsize=FONT['title'] + 1, fontweight='bold', y=1.01)

# plt.savefig('transport_timeseries_CORRECTED.png',
#             dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Load in SIC data 

In [ ]:
import glob

In [ ]:
files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

In [ ]:
files

In [ ]:
import glob
import xarray as xr

files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

sic = xr.open_mfdataset(
    files,
    combine="by_coords"
)

print(sic)

In [ ]:
one = xr.open_dataset(files[1])


In [ ]:
print(one)

In [ ]:
import xarray as xr
import pandas as pd
import glob
import re
import numpy as np

In [ ]:
def get_time_from_filename(fname):
    match = re.search(r'(\d{8})', fname)
    if match is None:
        raise ValueError(f"No date found in filename: {fname}")
    return pd.to_datetime(match.group(1), format='%Y%m%d')

In [ ]:
files = glob.glob(
    r"C:\Users\mblan\Documents\TNB_Research\data\SIC_Ross\**\*.nc",
    recursive=True
)

datasets = []

for f in files:
    ds = xr.open_dataset(f)

    # rename variable if needed
    ds = ds.rename({"z": "sic"})  # sea ice concentration

    # add time dimension
    time = get_time_from_filename(f)
    ds = ds.expand_dims({"time": [time]})

    datasets.append(ds)

In [ ]:
len(datasets)

In [ ]:
datasets[9]

In [ ]:
ds_all = xr.concat(datasets, dim="time")

In [ ]:
ds_all

In [ ]:
# rough Ross Sea bounds (you may tweak)
x_min, x_max = -2.2e6, 1.5e6
y_min, y_max = -2.2e6, -0.2e6

In [ ]:
ds_ross = ds_all.where(
    (ds_all.x >= x_min) &
    (ds_all.x <= x_max) &
    (ds_all.y >= y_min) &
    (ds_all.y <= y_max),
    drop=True
)

In [ ]:
ds_ross

In [ ]:
ds_ross.sic.isel(time=500).plot(
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100,
    figsize=(8,8)
)

plt.title(str(ds_ross.time.isel(time=0).values))
plt.show()

In [ ]:
x_min = 0.2e6
x_max = 0.9e6

y_min = -2.5e6
y_max = -1.6e6

In [ ]:
ds_tnb = ds_ross.where(
    (ds_ross.x >= x_min) &
    (ds_ross.x <= x_max) &
    (ds_ross.y >= y_min) &
    (ds_ross.y <= y_max),
    drop=True
)

In [ ]:
ds_tnb.sic.isel(time=500).plot(
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100,
    figsize=(8,8)
)

plt.title(str(ds_tnb.time.isel(time=0).values))
plt.show()

In [ ]:
ds_ross.to_netcdf('/Users/mblan/Documents/TNB_Research/data/SIC_Ross/SIC_Ross_2016_2019.nc')

In [ ]:
ds_ross = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/SIC_Ross/SIC_Ross_2016_2019.nc')

In [ ]:
ds_ross.polar_stereographic

In [ ]:
import pyproj

crs_ps = pyproj.CRS.from_cf(
    ds_ross.polar_stereographic.attrs
)

crs_ps

In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs(
    crs_ps,
    "EPSG:4326",
    always_xy=True
)

In [ ]:
import numpy as np

X, Y = np.meshgrid(
    ds_ross.x.values,
    ds_ross.y.values
)

lon, lat = transformer.transform(X, Y)

In [ ]:
ds_ross = ds_ross.assign_coords(
    lon=(("y","x"), lon),
    lat=(("y","x"), lat)
)

In [ ]:
ds_ross

In [ ]:
tnb = ds_ross.where(
    (ds_ross.lon >= 160) &
    (ds_ross.lon <= 168) &
    (ds_ross.lat >= -76) &
    (ds_ross.lat <= -73),
    drop=True
)

In [ ]:
tnb

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cmocean

fig = plt.figure(figsize=(8,8))

ax = plt.axes(
    projection=ccrs.SouthPolarStereo()
)

ax.coastlines()

tnb.sic.isel(time=0).plot(
    ax=ax,
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100
)

ax.set_extent(
    [160,168,-76,-73],
    crs=ccrs.PlateCarree()
)

plt.show()

In [ ]:
tnb.to_netcdf('/Users/mblan/Documents/TNB_Research/data/SIC_Ross/SIC_TNB_2016_2019.nc')

In [ ]:
# open TNB dataset 
tnb = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/SIC_Ross/SIC_TNB_2016_2019.nc')

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cmocean

fig = plt.figure(figsize=(8,8))

ax = plt.axes(
    projection=ccrs.SouthPolarStereo(
        central_longitude=165
    )
)

ax.coastlines()

tnb.sic.isel(time=0).plot(
    ax=ax,
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100
)

ax.set_extent(
    [160,168,-76,-73],
    crs=ccrs.PlateCarree()
)

plt.show()

In [ ]:
tnb

In [ ]:
tnb_anim = tnb.sel(time=slice('2018-07-01','2018-08-01'))

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import cmocean
import matplotlib.animation as animation


# Create figure
fig = plt.figure(figsize=(8,8))

ax = plt.axes(
    projection=ccrs.SouthPolarStereo(
        central_longitude=165
    )
)

ax.coastlines()

ax.set_extent(
    [160,168,-76,-73],
    crs=ccrs.PlateCarree()
)


# initial frame
sic_plot = tnb_anim.sic.isel(time=0).plot(
    ax=ax,
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100,
    add_colorbar=True
)

# title
title = ax.set_title(
    str(tnb_anim.time.isel(time=0).values)[:10]
)


# animation function
def update(frame):

    # get new SIC
    sic = tnb_anim.sic.isel(time=frame)

    # update color values
    sic_plot.set_array(
        sic.values.ravel()
    )

    # update title
    title.set_text(
        tnb_anim.time.isel(time=frame).dt.strftime("%Y-%m-%d").item()
    )

    return [sic_plot, title]


# create animation
ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(tnb_anim.time),
    interval=200,   # milliseconds between frames
    blit=False
)


# save mp4
ani.save(
    "TNB_SIC_animation_July.mp4",
    writer="ffmpeg",
    fps=5,
    dpi=200
)


plt.close()

In [ ]:
!conda install -c conda-forge ffmpeg

In [ ]:
tnb_zoomed = tnb.where(
    (tnb.lon >= 163) &
    (tnb.lon <= 166) &
    (tnb.lat >= -75.5) &
    (tnb.lat <= -74.7),
    drop=True
)

In [ ]:
fig = plt.figure(figsize=(8,8))

ax = plt.axes(
    projection=ccrs.SouthPolarStereo(
        central_longitude=165
    )
)

ax.coastlines()

tnb_zoomed.sic.isel(time=500).plot(
    ax=ax,
    x="lon",
    y="lat",
    transform=ccrs.PlateCarree(),
    cmap=cmocean.cm.ice,
    vmin=0,
    vmax=100
)

ax.set_extent(
    [160,168,-76,-73],
    crs=ccrs.PlateCarree()
)

plt.show()

In [ ]:
# average TNB zoomed SIC to make timeseries 
tnb_SIC_ts = tnb_zoomed.sic.mean('x').mean('y')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tnb_SIC_ts.time,tnb_SIC_ts)

In [ ]:
# crop to same time as rest of data 
tnb_sic_2018 = tnb_SIC_ts.sel(time=slice('2018-03-14','2019-01-01'))

In [ ]:
tnb_sic_2018

In [ ]:
# define rolling average 
rolling = 96 # daily 
# plot timeseries of along trough velocity, wind speed, salinity, temperature, density, SIC 
fig, ax = plt.subplots(nrows=6,ncols=1,figsize=(10,10),sharex=True)

# WIND SPEED 
ax[0].plot(wspd_15.rolling(time=rolling).mean().time,wspd_15.rolling(time=rolling).mean().wind_speed)
ax[0].set_ylabel('Wind Speed (m/s)')
ax[0].tick_params(labelbottom=False)

# SIC 
ax[1].plot(tnb_sic_2018.time,tnb_sic_2018)
ax[1].set_ylabel('SIC (%)')

# Transport
ax[2].plot(Q_TNBD_along.rolling(time=rolling).mean().time,Q_TNBD_along.rolling(time=rolling).mean()*1e-6)
ax[2].axhline(y=0, color="k", linestyle="--", linewidth=2)
ax[2].set_ylabel('Along-Trough \n Transport (Sv)')
ax[2].tick_params(labelbottom=False)

# Absolute Salinity 
ax[3].plot(TNBD_ctd.rolling(time=rolling).mean().time,TNBD_ctd.rolling(time=rolling).mean().abs_sal)
ax[3].set_ylabel('Absolute \n Salinity (g/kg)')
ax[3].tick_params(labelbottom=False)

# Potential Temp 
ax[4].plot(TNBD_ctd.rolling(time=rolling).mean().time,TNBD_ctd.rolling(time=rolling).mean().theta)
ax[4].set_ylabel('Potential \n Temp (Deg C)')
ax[4].tick_params(labelbottom=False)

# potential density anomaly 
ax[5].plot(TNBD_ctd.rolling(time=rolling).mean().time,TNBD_ctd.rolling(time=rolling).mean().sigma0)
ax[5].set_ylabel('Potential Density \n Anomaly (kg/m3)')

# Loop through each axes object to enable vertical grids
for axs in ax:
    # axs.plot([1, 2, 3], [4, 5, 6])
    axs.grid(True, axis='both', color='gray', linestyle='--', linewidth=0.5)

plt.suptitle('TNBD')

In [ ]:
# define rolling average 
rolling = 96 # daily 
# plot timeseries of along trough velocity, wind speed, salinity, temperature, density, SIC 
fig, ax = plt.subplots(nrows=6,ncols=1,figsize=(10,10),sharex=True)

# WIND SPEED 
ax[0].plot(wspd_15.rolling(time=rolling).mean().time,wspd_15.rolling(time=rolling).mean().wind_speed)
ax[0].set_ylabel('Wind Speed (m/s)')
ax[0].tick_params(labelbottom=False)

# SIC 
ax[1].plot(tnb_sic_2018.time,tnb_sic_2018)
ax[1].set_ylabel('SIC (%)')

# Along Trough Velocity 
ax[2].plot(Q_DITD_along.rolling(time=rolling).mean().time,Q_DITD_along.rolling(time=rolling).mean()*1e-6)
ax[2].axhline(y=0, color="k", linestyle="--", linewidth=2)
ax[2].set_ylabel('Along-Trough \n Transport (Sv)')
ax[2].tick_params(labelbottom=False)

# Absolute Salinity 
ax[3].plot(DITD_ctd.rolling(time=rolling).mean().time,DITD_ctd.rolling(time=rolling).mean().abs_sal)
ax[3].set_ylabel('Absolute \n Salinity (g/kg)')
ax[3].tick_params(labelbottom=False)

# Potential Temp 
ax[4].plot(DITD_ctd.rolling(time=rolling).mean().time,DITD_ctd.rolling(time=rolling).mean().theta)
ax[4].set_ylabel('Potential \n Temp (Deg C)')
ax[4].tick_params(labelbottom=False)

# potential density anomaly 
ax[5].plot(DITD_ctd.rolling(time=rolling).mean().time,DITD_ctd.rolling(time=rolling).mean().sigma0)
ax[5].set_ylabel('Potential Density \n Anomaly (kg/m3)')

# Loop through each axes object to enable vertical grids
for axs in ax:
    # axs.plot([1, 2, 3], [4, 5, 6])
    axs.grid(True, axis='both', color='gray', linestyle='--', linewidth=0.5)

# plt.suptitle('TNBD')

### T/S Diagram - Quals Paper 

In [ ]:
# # need to get all datasets on same time and sampling interval 

# time_slice = slice('2018-03-14','2018-12-31')
# resample = '30min'

In [ ]:
# TNBD_ctd_top = TNBD_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# TNBD_ctd_bot = TNBD_ctd_bot.sel(time=time_slice).resample(time=resample).mean()
# TNBD_vel_top = TNBD_vel_top.sel(time=time_slice).resample(time=resample).mean()
# TNBD_vel_bot = TNBD_vel_bot.sel(time=time_slice).resample(time=resample).mean()

# DITD_ctd_top = DITD_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# DITD_ctd_bot = DITD_ctd_bot.sel(time=time_slice).resample(time=resample).mean()
# DITD_vel_top = DITD_vel_top.sel(time=time_slice).resample(time=resample).mean()
# DITD_vel_bot = DITD_vel_bot.sel(time=slice('2018-03-14','2018-06')).resample(time=resample).mean()       # sensor cut out in june 2018

# D_ctd_top = D_ctd_top.sel(time=time_slice).resample(time=resample).mean()
# D_ctd_mid = D_ctd_mid.sel(time=time_slice).resample(time=resample).mean()
# # D_ctd_bot = D_ctd_bot.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out in 2017

# # D_vel_top = D_vel_top.sel(time=time_slice).resample(time=resample).mean()        # this sensor cut out end of 2017
# D_vel_mid = D_vel_mid.sel(time=time_slice).resample(time=resample).mean()
# D_vel_bot = D_vel_bot.sel(time=time_slice).resample(time=resample).mean()

In [ ]:
TNBD_ctd_top['abs_sal'] = gsw.SA_from_SP(TNBD_ctd_top.sal,1000,TNBD_ctd_top.lon,TNBD_ctd_top.lat)

In [ ]:
TNBD_ctd_bot['abs_sal'] = gsw.SA_from_SP(TNBD_ctd_bot.sal,1100,TNBD_ctd_bot.lon,TNBD_ctd_bot.lat)
DITD_ctd_bot['abs_sal'] = gsw.SA_from_SP(DITD_ctd_bot.sal,1200,DITD_ctd_bot.lon,DITD_ctd_bot.lat)
DITD_ctd_top['abs_sal'] = gsw.SA_from_SP(DITD_ctd_top.sal,1100,DITD_ctd_top.lon,DITD_ctd_top.lat)

D_ctd_top['abs_sal'] = gsw.SA_from_SP(D_ctd_top.sal,556,D_ctd_top.lon,D_ctd_top.lat)
D_ctd_mid['abs_sal'] = gsw.SA_from_SP(D_ctd_mid.sal,772,D_ctd_mid.lon,D_ctd_mid.lat)

In [ ]:
TNBD_ctd_top['theta'] = gsw.pt0_from_t(TNBD_ctd_top.abs_sal,TNBD_ctd_top.temp,1000)
TNBD_ctd_bot['theta'] = gsw.pt0_from_t(TNBD_ctd_bot.abs_sal,TNBD_ctd_bot.temp,1100)

DITD_ctd_top['theta'] = gsw.pt0_from_t(DITD_ctd_top.abs_sal,DITD_ctd_top.temp,1100)
DITD_ctd_bot['theta'] = gsw.pt0_from_t(DITD_ctd_bot.abs_sal,DITD_ctd_bot.temp,1200)

D_ctd_top['theta'] = gsw.pt0_from_t(D_ctd_top.abs_sal,D_ctd_top.temp,556)
D_ctd_mid['theta'] = gsw.pt0_from_t(D_ctd_mid.abs_sal,D_ctd_mid.temp,772)

In [ ]:
import gsw

In [ ]:
def TS_template(ax1):
    """ 
    """
    # Create figure
    # fig = plt.figure(figsize=(10, 6),constrained_layout=True)
    # # Top left square
    # ax1 = plt.subplot(1,1,1)

    mint=-2.3
    maxt=-1.4
    mins=34.4
    maxs=36

    tempL=np.linspace(mint,maxt)
    salL=np.linspace(mins,maxs)
    Tg, Sg = np.meshgrid(tempL,salL)

    lon = 165
    lat = -75.0112
    p = 1093
    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    # rho = gsw.rho(SA, Tg, 0)
    # sigma_theta = rho - 1000

    # CALCULATE DENSITY (METHOD 2)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing1100 = gsw.t_freezing(gsw.SA_from_SP(salL, 0, lon, lat), p=0, saturation_fraction=0)
    plt.plot(salL, t_freezing1100, 'k--', label='Freezing Line at 0 dbar')

    cs = ax1.contour(Sg, Tg, sigma_theta, colors='k', zorder=1, alpha=0.7, levels=4)
    cl=plt.clabel(cs,fontsize=10,inline=True,manual=[(34.45,-2.2),(34.55,-2.05),(34.75,-1.75)])

In [ ]:
import numpy as np
import gsw
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe

def TS_template(ax, mint, maxt, mins, maxs, lon=165, lat=-75.0112, p=1093,
                n_levels=6, freezing_p=0, manual_labels=None):
    """
    Draws sigma-theta density contours and the surface freezing line on ax.
    Generalized from your original so it can be reused at different zoom
    levels (overview panel vs. per-mooring zoomed panels).
    """
    tempL = np.linspace(mint, maxt, 200)
    salL  = np.linspace(mins, maxs, 200)
    Tg, Sg = np.meshgrid(tempL, salL)

    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing = gsw.t_freezing(gsw.SA_from_SP(salL, freezing_p, lon, lat),
                                  p=freezing_p, saturation_fraction=0)
    ax.plot(salL, t_freezing, 'k--', linewidth=1, zorder=1,
            label='Surface freezing line')

    cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', linewidths=0.8,
                     alpha=0.8, levels=n_levels, zorder=1)
    if manual_labels is not None:
        ax.clabel(cs, fontsize=8, inline=True, manual=manual_labels, fmt='%.2f')
    else:
        ax.clabel(cs, fontsize=8, inline=True, fmt='%.2f')

    ax.set_xlim(mins, maxs)
    ax.set_ylim(mint, maxt)
    return cs

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
ax1 = axes[0, 0]
axes_row2 = [axes[0, 1], axes[1, 0], axes[1, 1]]

# ── Panel a: overview, all moorings, all depths, zoomed out ────────────────
TS_template(ax1, mint=-2.4, maxt=1.5, mins=33.5, maxs=35.5)

ax1.scatter(TNBD_ctd_top.abs_sal, TNBD_ctd_top.theta, s=0.5, color='steelblue', label='TNBD')
ax1.scatter(TNBD_ctd_bot.abs_sal, TNBD_ctd_bot.theta, s=0.5, color='steelblue')

ax1.scatter(DITD_ctd_top.abs_sal, DITD_ctd_top.theta, s=0.5, color='tomato', label='DITD')
ax1.scatter(DITD_ctd_bot.abs_sal, DITD_ctd_bot.theta, s=0.5, color='tomato')

ax1.scatter(D_ctd_top.abs_sal, D_ctd_top.theta, s=0.5, color='tab:green', label='D')
ax1.scatter(D_ctd_mid.abs_sal, D_ctd_mid.theta, s=0.5, color='tab:green')

# Water mass labels — approximate placeholders; nudge once you see your data
water_masses = {
    'ASW':  (34,  -0.5),
    'HSSW': (35, -1.8),
    'ISW':  (34.55, -2.1),
    'mCDW': (34.65, -0.3),
    'CDW':  (34.85,  1.2),
}
for name, (s, t) in water_masses.items():
    ax1.annotate(name, xy=(s, t), fontsize=9, fontweight='bold', color='k',
                 ha='center', va='center',
                 path_effects=[pe.withStroke(linewidth=2.5, foreground='white')])

ax1.set_xlabel('Absolute Salinity (g/kg)')
ax1.set_ylabel('Potential Temperature (°C)')
ax1.set_title('a) All moorings', loc='left', fontsize=10, weight='bold')
ax1.legend(markerscale=10, fontsize=8, loc='upper left', framealpha=0.9)

# ── Panels b–d: per-mooring, deepest (~1100 m) sensor, colored by time ─────
deep_sensors = {
    'TNBD': TNBD_ctd_bot,
    'DITD': DITD_ctd_top,
    'D':    D_ctd_mid,
}

# Shared time normalization so color scale is consistent across all 3 panels
all_times  = np.concatenate([mdates.date2num(ds.time.values) for ds in deep_sensors.values()])
tmin, tmax = all_times.min(), all_times.max()

# Zoomed bounds — placeholder, tighten/loosen once you see your actual
# near-1100 m T/S spread
panel_bounds = dict(mint=-1.94, maxt=-1.9, mins=34.75, maxs=35.2)

labels = ['b', 'c', 'd']

for ax, (name, ds), lbl in zip(axes_row2, deep_sensors.items(), labels):
    TS_template(ax, **panel_bounds)
    tvals = mdates.date2num(ds.time.values)
    sc = ax.scatter(ds.abs_sal, ds.theta, c=tvals, cmap='viridis',
                     vmin=tmin, vmax=tmax, s=4)
    ax.set_title(f'{lbl}) {name}', loc='left', fontsize=10,weight='bold')
    ax.set_xlabel('Absolute Salinity (g/kg)')

# Only label y-axis on the left column to reduce clutter
axes[0, 1].set_ylabel('')
axes[1, 1].set_ylabel('')
axes[1, 0].set_ylabel('Potential Temperature (°C)')

# Shared horizontal colorbar under the bottom row, formatted as dates
cbar = fig.colorbar(sc, ax=axes_row2, orientation='horizontal',
                     fraction=0.05, pad=0.12, aspect=40)
cbar.ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
cbar.ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
cbar.set_label('Time')

for ax in [ax1, *axes_row2]:
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_ha('right')

# plt.savefig('TS_diagram_Quals_redo.png',dpi=300,bbox_inches='tight',facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(6, 5))

# ── T/S template (isopycnals, freezing line, etc.) ───────────────────────────
TS_template(ax, mint=-2.4, maxt=1.5, mins=33.5, maxs=35.5)

# ── Scatter data ──────────────────────────────────────────────────────────────
ax.scatter(TNBD_ctd_top.abs_sal, TNBD_ctd_top.theta,
           s=0.5, color='steelblue', label='TNBD')
ax.scatter(TNBD_ctd_bot.abs_sal, TNBD_ctd_bot.theta,
           s=0.5, color='steelblue')

ax.scatter(DITD_ctd_top.abs_sal, DITD_ctd_top.theta,
           s=0.5, color='tomato', label='DITD')
ax.scatter(DITD_ctd_bot.abs_sal, DITD_ctd_bot.theta,
           s=0.5, color='tomato')

ax.scatter(D_ctd_top.abs_sal, D_ctd_top.theta,
           s=0.5, color='tab:green', label='D')
ax.scatter(D_ctd_mid.abs_sal, D_ctd_mid.theta,
           s=0.5, color='tab:green')

# ── Water mass labels ─────────────────────────────────────────────────────────
water_masses = {
    'ASW':  (34.0,   -0.5),
    'HSSW': (35.0,   -1.8),
    'ISW':  (34.55,  -2.1),
    'mCDW': (34.65,  -0.3),
    'CDW':  (34.85,   1.2),
}
for name, (s, t) in water_masses.items():
    ax.annotate(name, xy=(s, t), fontsize=9,
                fontweight='bold', color='k',
                ha='center', va='center',
                path_effects=[
                    pe.withStroke(linewidth=2.5, foreground='white')])

# ── Labels and formatting ─────────────────────────────────────────────────────
ax.set_xlabel('Absolute Salinity (g/kg)', fontsize=11)
ax.set_ylabel('Potential Temperature (°C)', fontsize=11)
ax.set_title('All moorings', loc='left', fontsize=11, fontweight='bold')
ax.legend(markerscale=10, fontsize=9, loc='upper left', framealpha=0.9)
ax.tick_params(labelsize=10)

for label in ax.get_xticklabels():
    label.set_rotation(30)
    label.set_ha('right')

plt.tight_layout()
# plt.savefig('TS_diagram_all_moorings.png',
#             dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import numpy as np

# ── ADJUSTABLE TIME WINDOWS ───────────────────────────────────────────────────
WIN1_START = '2018-08-15'
WIN1_END   = '2018-10-01'
WIN2_START = '2018-11-01'
WIN2_END   = '2018-12-31'

# ── MOORING DATA ──────────────────────────────────────────────────────────────
MOORINGS_TS = {
    'TNBD': {'ds': TNBD_ctd_bot, 'color': 'steelblue'},
    'D':    {'ds': D_ctd_mid,    'color': 'seagreen'},
    'DITD': {'ds': DITD_ctd_top, 'color': 'tomato'},
}

# ── TS DIAGRAM BOUNDS PER MOORING (zoom in to show seasonal spread) ───────────
# Adjust these once you see your data
TS_BOUNDS = {
    'TNBD': dict(mint=-1.95, maxt=-1.88, mins=34.78, maxs=35.05),
    'D':    dict(mint=-1.95, maxt=-1.88, mins=34.78, maxs=35.05),
    'DITD': dict(mint=-1.95, maxt=-1.88, mins=34.78, maxs=35.05),
}

# Shared time normalization for colorbar
all_times = np.concatenate([
    mdates.date2num(props['ds'].time.values)
    for props in MOORINGS_TS.values()
])
tmin, tmax = all_times.min(), all_times.max()

# ── FIGURE SETUP ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 13),
                          constrained_layout=True)

col_titles = [f'Timeseries {WIN1_START} – {WIN1_END}',
              'T/S diagram (full record, colored by time)',
              f'Timeseries {WIN2_START} – {WIN2_END}']
row_titles = list(MOORINGS_TS.keys())

for row, (name, props) in enumerate(MOORINGS_TS.items()):
    ds    = props['ds']
    color = props['color']

    # ── Column 0: timeseries window 1 ────────────────────────────────────────
    ax   = axes[row, 0]
    ax2  = ax.twinx()
    ds_w1 = ds.sel(time=slice(WIN1_START, WIN1_END))

    l1, = ax.plot(ds_w1.time.values, ds_w1['theta'].values,
                  color=color, lw=1.5, label='θ (°C)')
    l2, = ax2.plot(ds_w1.time.values, ds_w1['abs_sal'].values,
                   color=color, lw=1.5, ls='--', alpha=0.7,
                   label='S (g/kg)')

    ax.set_ylabel('Pot. temp. (°C)', color=color, fontsize=9)
    ax2.set_ylabel('Abs. sal. (g/kg)', color=color, fontsize=9)
    ax.tick_params(axis='y', colors=color, labelsize=8)
    ax2.tick_params(axis='y', colors=color, labelsize=8)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.spines['top'].set_visible(False)

    lines  = [l1, l2]
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, fontsize=7, loc='best', framealpha=0.8)

    # Row label on left
    ax.set_ylabel(f'{name}\nPot. temp. (°C)', color=color, fontsize=9,
                  fontweight='bold')

    # ── Column 1: T/S diagram colored by time ────────────────────────────────
    ax = axes[row, 1]
    TS_template(ax, **TS_BOUNDS[name])

    tvals = mdates.date2num(ds.time.values)
    sc    = ax.scatter(ds['abs_sal'].values, ds['theta'].values,
                       c=tvals, cmap='viridis',
                       vmin=tmin, vmax=tmax, s=2, zorder=3)

    ax.set_xlabel('Abs. salinity (g/kg)', fontsize=9)
    ax.set_ylabel('Pot. temp. (°C)', fontsize=9)
    ax.tick_params(labelsize=8)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(30); lbl.set_ha('right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # ── Column 2: timeseries window 2 ────────────────────────────────────────
    ax  = axes[row, 2]
    ax2 = ax.twinx()
    ds_w2 = ds.sel(time=slice(WIN2_START, WIN2_END))

    ax.plot(ds_w2.time.values, ds_w2['theta'].values,
            color=color, lw=1.5, label='θ (°C)')
    ax2.plot(ds_w2.time.values, ds_w2['abs_sal'].values,
             color=color, lw=1.5, ls='--', alpha=0.7,
             label='S (g/kg)')

    ax.set_ylabel('Pot. temp. (°C)', color=color, fontsize=9)
    ax2.set_ylabel('Abs. sal. (g/kg)', color=color, fontsize=9)
    ax.tick_params(axis='y', colors=color, labelsize=8)
    ax2.tick_params(axis='y', colors=color, labelsize=8)
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.spines['top'].set_visible(False)

# ── Column titles on top row ──────────────────────────────────────────────────
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9, fontweight='bold', pad=6)

# ── Shared colorbar beneath middle column ─────────────────────────────────────
cbar = fig.colorbar(sc, ax=axes[:, 1],
                     orientation='vertical',
                     fraction=0.04, pad=0.02, aspect=30)
cbar.ax.yaxis.set_major_locator(mdates.MonthLocator(interval=1))
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
cbar.set_label('Time (2018)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# ── Panel labels ──────────────────────────────────────────────────────────────
panel_labels = ['a','b','c','d','e','f','g','h','i']
for i, ax in enumerate(axes.flat):
    ax.text(0.02, 0.97, f'{panel_labels[i]})',
            transform=ax.transAxes, fontsize=9,
            fontweight='bold', va='top')

# plt.savefig('seasonal_TS_variability.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ── SETTINGS ──────────────────────────────────────────────────────────────────
# Adjust dataset and time windows to your data
MOORING_NAME = 'TNBD'
ds_ctd       = TNBD_ctd_bot   # dataset with abs_sal and theta
COLOR        = 'steelblue'

# Time windows to annotate — adjust to where you see transitions
TRANSITIONS = {
    'HSSW\narrival':  ('2018-09-01', '2018-09-30', 'tomato'),
    'Post-HSSW':      ('2018-10-01', '2018-11-30', 'darkorange'),
}

# ── FIGURE: two-panel layout ──────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 9))
gs  = fig.add_gridspec(2, 2,
                        width_ratios=[1.6, 1],
                        height_ratios=[1, 1],
                        hspace=0.35, wspace=0.35)

ax_T  = fig.add_subplot(gs[0, 0])   # temperature timeseries
ax_S  = fig.add_subplot(gs[1, 0],   # salinity timeseries
                         sharex=ax_T)
ax_TS = fig.add_subplot(gs[:, 1])   # T/S trajectory

# ── PANEL LEFT: T and S timeseries ───────────────────────────────────────────
time   = ds_ctd.time.values
theta  = ds_ctd['theta'].values
absal  = ds_ctd['abs_sal'].values

# Temperature
ax_T.plot(time, theta, color=COLOR, lw=1.2, alpha=0.6)
ax_T.set_ylabel('Conservative\ntemperature (°C)', fontsize=11)
ax_T.tick_params(labelsize=10)
ax_T.spines['top'].set_visible(False)
ax_T.spines['right'].set_visible(False)
ax_T.text(0.01, 0.95, 'a) Temperature', transform=ax_T.transAxes,
          fontsize=10, fontweight='bold', va='top')

# Salinity
ax_S.plot(time, absal, color=COLOR, lw=1.2, alpha=0.6)
ax_S.set_ylabel('Absolute\nsalinity (g/kg)', fontsize=11)
ax_S.tick_params(labelsize=10)
ax_S.spines['top'].set_visible(False)
ax_S.spines['right'].set_visible(False)
ax_S.text(0.01, 0.95, 'b) Salinity', transform=ax_S.transAxes,
          fontsize=10, fontweight='bold', va='top')

# Shade transition windows on both timeseries panels
for label, (t0, t1, c) in TRANSITIONS.items():
    for ax in [ax_T, ax_S]:
        ax.axvspan(np.datetime64(t0), np.datetime64(t1),
                   color=c, alpha=0.15, zorder=0)
    # Label on temperature panel only
    mid = np.datetime64(t0) + (np.datetime64(t1) -
                                np.datetime64(t0)) / 2
    ax_T.text(mid, ax_T.get_ylim()[1] if ax_T.get_ylim()[1] != 1.0
              else np.nanmax(theta),
              label, ha='center', va='bottom', fontsize=8,
              color=c, fontweight='bold',
              path_effects=[pe.withStroke(linewidth=2,
                                           foreground='white')])

# x-axis formatting on bottom panel only
ax_S.xaxis.set_major_locator(mdates.MonthLocator())
ax_S.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_S.tick_params(axis='x', rotation=30, labelsize=10)
ax_T.tick_params(axis='x', labelbottom=False)

# ── PANEL RIGHT: T/S trajectory colored by month ─────────────────────────────
# Color each point by month so seasonal trajectory is clear
time_num = mdates.date2num(time)
months   = pd.DatetimeIndex(time).month

# Use a cyclic colormap — one color per month
month_colors = cm.get_cmap('tab10', 12)
norm         = mcolors.BoundaryNorm(np.arange(0.5, 13.5, 1),
                                     month_colors.N)

sc = ax_TS.scatter(absal, theta,
                    c=months, cmap='tab10', norm=norm,
                    s=3, alpha=0.6, zorder=2)

# Draw monthly mean trajectory with arrows to show direction of travel
df = pd.DataFrame({'sal': absal, 'temp': theta,
                    'month': months},
                   index=pd.DatetimeIndex(time))
monthly_mean = df.groupby('month')[['sal', 'temp']].mean()

# Plot trajectory line connecting monthly means
ax_TS.plot(monthly_mean['sal'], monthly_mean['temp'],
           color='k', lw=1.0, alpha=0.4, zorder=3)

# Arrows showing direction of travel between consecutive months
for i in range(len(monthly_mean) - 1):
    s0 = monthly_mean['sal'].iloc[i]
    t0 = monthly_mean['temp'].iloc[i]
    ds = monthly_mean['sal'].iloc[i+1]  - s0
    dt = monthly_mean['temp'].iloc[i+1] - t0
    ax_TS.annotate('',
                   xy=(s0 + ds, t0 + dt),
                   xytext=(s0 + ds*0.4, t0 + dt*0.4),
                   arrowprops=dict(arrowstyle='->', color='k',
                                   lw=1.2, mutation_scale=10),
                   zorder=4)

# Monthly mean markers labeled by month abbreviation
month_abbr = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
for month, row in monthly_mean.iterrows():
    c_m = month_colors(month - 1)
    ax_TS.plot(row['sal'], row['temp'],
               'o', ms=8, color=c_m,
               markeredgecolor='white', markeredgewidth=0.8,
               zorder=5)
    ax_TS.annotate(month_abbr[month-1],
                   xy=(row['sal'], row['temp']),
                   xytext=(4, 4), textcoords='offset points',
                   fontsize=7, fontweight='bold', color=c_m,
                   path_effects=[pe.withStroke(linewidth=1.5,
                                                foreground='white')],
                   zorder=6)

# Shade transition region on T/S diagram
for label, (t0, t1, c) in TRANSITIONS.items():
    mask = ((pd.DatetimeIndex(time) >= t0) &
            (pd.DatetimeIndex(time) <= t1))
    if mask.sum() > 0:
        ax_TS.scatter(absal[mask], theta[mask],
                      color=c, s=8, alpha=0.5, zorder=4,
                      label=label)

# T/S template (isopycnals etc.)
TS_template(ax_TS,
            mint=np.nanmin(theta) - 0.02,
            maxt=np.nanmax(theta) + 0.02,
            mins=np.nanmin(absal) - 0.05,
            maxs=np.nanmax(absal) + 0.05)

ax_TS.set_xlabel('Absolute salinity (g/kg)', fontsize=11)
ax_TS.set_ylabel('Conservative temperature (°C)', fontsize=11)
ax_TS.tick_params(labelsize=10)
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)
ax_TS.text(0.01, 0.99, 'c) T/S trajectory',
           transform=ax_TS.transAxes, fontsize=10,
           fontweight='bold', va='top')

for lbl in ax_TS.get_xticklabels():
    lbl.set_rotation(30); lbl.set_ha('right')

# ── Colorbar for monthly T/S scatter ─────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax_TS, orientation='vertical',
                     fraction=0.04, pad=0.02, aspect=25)
cbar.set_ticks(np.arange(1, 13))
cbar.set_ticklabels(month_abbr, fontsize=8)
cbar.set_label('Month', fontsize=9)

# ── Transition legend ─────────────────────────────────────────────────────────
handles = [plt.Rectangle((0,0), 1, 1, color=c, alpha=0.4, label=lbl)
           for lbl, (_, _, c) in TRANSITIONS.items()]
ax_T.legend(handles=handles, fontsize=8, loc='lower right',
             framealpha=0.8)

fig.suptitle(f'{MOORING_NAME} — seasonal T/S evolution 2018',
             fontsize=13, fontweight='bold')

# plt.savefig(f'seasonal_TS_evolution_{MOORING_NAME}.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS — adjust these to zoom into phase transitions
# ══════════════════════════════════════════════════════════════════════════════

MOORING_NAME = 'TNBD'
ds_ctd       = TNBD_ctd_bot   # dataset with abs_sal and theta
COLOR        = 'steelblue'

# Two ~2-week zoom windows — adjust to where you see the phase transitions
WIN1_START = '2018-09-05'
WIN1_END   = '2018-09-21'   # T/S going OUT of phase

WIN2_START = '2018-11-01'
WIN2_END   = '2018-11-15'   # T/S going back IN phase

# Phase transition markers on the T/S diagram
# Set these to the exact timestamps where the slope changes
OUT_OF_PHASE_TIME = '2018-09-10T16:40'   # when T/S decouple
IN_PHASE_TIME     = '2018-11-06T11:30'   # when T/S re-couple

# ══════════════════════════════════════════════════════════════════════════════

# ── Prep ──────────────────────────────────────────────────────────────────────
time  = ds_ctd.time.values
theta = ds_ctd['theta'].values
absal = ds_ctd['abs_sal'].values

time_num = mdates.date2num(time)
tmin, tmax = time_num.min(), time_num.max()

# Find nearest index to each marker time
def nearest_idx(time_arr, t_str):
    return np.argmin(np.abs(time_arr - np.datetime64(t_str)))

idx_out = nearest_idx(time, OUT_OF_PHASE_TIME)
idx_in  = nearest_idx(time, IN_PHASE_TIME)

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 8))
gs  = fig.add_gridspec(2, 2,
                        width_ratios=[1.3, 1],
                        height_ratios=[1, 1],
                        hspace=0.38, wspace=0.38)

ax_w1 = fig.add_subplot(gs[0, 0])   # top left: zoom window 1
ax_w2 = fig.add_subplot(gs[1, 0])   # bottom left: zoom window 2
ax_TS = fig.add_subplot(gs[:, 1])   # right: T/S diagram (full record)


def plot_dual_timeseries(ax, ds, t_start, t_end, title):
    """
    Double y-axis timeseries of temperature and salinity
    in distinct colors with solid lines.
    """
    ds_win = ds.sel(time=slice(t_start, t_end))
    t      = ds_win.time.values
    temp   = ds_win['theta'].values
    sal    = ds_win['abs_sal'].values

    ax2 = ax.twinx()

    # Temperature — tomato
    l1, = ax.plot(t, temp, color='tomato', lw=1.8, label='θ (°C)')
    ax.set_ylabel('Potential (°C)',
                  color='tomato', fontsize=10)
    ax.tick_params(axis='y', colors='tomato', labelsize=9)

    # Salinity — steelblue, solid line
    l2, = ax2.plot(t, sal, color='steelblue', lw=1.8, label='S (g/kg)')
    ax2.set_ylabel('Absolute\nsalinity (g/kg)',
                   color='steelblue', fontsize=10)
    ax2.tick_params(axis='y', colors='steelblue', labelsize=9)

    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.tick_params(axis='x', rotation=30, labelsize=9)

    lines  = [l1, l2]
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, fontsize=8, loc='best', framealpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False)

    return ax2


# ── Left panels: zoomed timeseries ───────────────────────────────────────────
plot_dual_timeseries(ax_w1, ds_ctd,
                     WIN1_START, WIN1_END,
                     f'a) T/S going out of phase\n'
                     f'   {WIN1_START} – {WIN1_END}')

plot_dual_timeseries(ax_w2, ds_ctd,
                     WIN2_START, WIN2_END,
                     f'b) T/S returning in phase\n'
                     f'   {WIN2_START} – {WIN2_END}')

# Mark the exact transition moment on each panel
for ax in [ax_w1]:
    ax.axvline(np.datetime64(OUT_OF_PHASE_TIME),
               color='k', lw=1.8, ls='--', zorder=5,
               label='Out of phase')
    ax.legend(fontsize=8, loc='best', framealpha=0.85)

for ax in [ax_w2]:
    ax.axvline(np.datetime64(IN_PHASE_TIME),
               color='seagreen', lw=1.8, ls='--', zorder=5,
               label='In phase')
    ax.legend(fontsize=8, loc='lower right', framealpha=0.85)

# ── Right panel: T/S diagram, full record, colored by time ───────────────────
# Make it square
ax_TS.set_box_aspect(1)

# Scatter colored by time
sc = ax_TS.scatter(absal, theta,
                    c=time_num, cmap='plasma',
                    vmin=tmin, vmax=tmax,
                    s=2, alpha=0.5, zorder=2)

# T/S template (isopycnals, freezing line)
TS_template(ax_TS,
            mint=float(np.nanmin(theta)) - 0.005,
            maxt=float(np.nanmax(theta)) + 0.005,
            mins=float(np.nanmin(absal)) - 0.01,
            maxs=float(np.nanmax(absal)) + 0.01)

# ── Phase transition markers ──────────────────────────────────────────────────
# Out of phase marker
ax_TS.plot(absal[idx_out], theta[idx_out],
           marker='*', ms=14, color='k',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6, label=f'Out of phase\n({OUT_OF_PHASE_TIME})')
# ax_TS.annotate(f'Out of phase\n{OUT_OF_PHASE_TIME}',
#                xy=(absal[idx_out], theta[idx_out]),
#                xytext=(8, 8), textcoords='offset points',
#                fontsize=8, color='tomato', fontweight='bold',
#                path_effects=[pe.withStroke(linewidth=2,
#                                             foreground='white')],
#                zorder=7)

# In phase marker
ax_TS.plot(absal[idx_in], theta[idx_in],
           marker='*', ms=14, color='seagreen',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6, label=f'Back in phase\n({IN_PHASE_TIME})')
# ax_TS.annotate(f'Back in phase\n{IN_PHASE_TIME}',
#                xy=(absal[idx_in], theta[idx_in]),
#                xytext=(8, -16), textcoords='offset points',
#                fontsize=8, color='seagreen', fontweight='bold',
#                path_effects=[pe.withStroke(linewidth=2,
#                                             foreground='white')],
#                zorder=7)

ax_TS.set_xlabel('Absolute salinity (g/kg)', fontsize=11)
ax_TS.set_ylabel('Potential temperature (°C)', fontsize=11)
ax_TS.tick_params(labelsize=10)
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)
ax_TS.text(0.02, 0.98, 'c) T/S diagram — full record',
           transform=ax_TS.transAxes, fontsize=10,
           fontweight='bold', va='top')

for lbl in ax_TS.get_xticklabels():
    lbl.set_rotation(30); lbl.set_ha('right')

# Legend for markers
ax_TS.legend(fontsize=8, loc='lower right',
              framealpha=0.85, markerscale=0.8)

# ── Colorbar beneath T/S diagram ──────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax_TS,
                     orientation='horizontal',
                     fraction=0.04, pad=0.14, aspect=30)
cbar.ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
cbar.ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
cbar.set_label('Time (2018)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# ── Title ─────────────────────────────────────────────────────────────────────
fig.suptitle(f'{MOORING_NAME} — T/S phase transitions 2018',
             fontsize=13, fontweight='bold')

# plt.savefig(f'TS_phase_transitions_{MOORING_NAME}.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── ADJUSTABLE INPUTS ─────────────────────────────────────────────────────────
ds_ctd        = TNBD_ctd_bot
RESAMPLE_DAYS = 1               # resample window in days
SHADE_START   = '2018-09-10'   # start of out-of-phase region
SHADE_END     = '2018-11-06'   # end of out-of-phase region

# ── RESAMPLE TO N-DAY MEANS ───────────────────────────────────────────────────
rule       = f'{RESAMPLE_DAYS}D'
theta_rs   = ds_ctd['theta'].resample(time=rule).mean()
sal_rs     = ds_ctd['abs_sal'].resample(time=rule).mean()

t    = theta_rs.time.values
temp = theta_rs.values
sal  = sal_rs.values

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax2 = ax.twinx()

# Temperature
l1, = ax.plot(t, temp, color='tomato', lw=1.8, label='θ (°C)')
ax.set_ylabel('Conservative temperature (°C)', color='tomato', fontsize=11)
ax.tick_params(axis='y', colors='tomato', labelsize=10)

# Salinity
l2, = ax2.plot(t, sal, color='steelblue', lw=1.8, label='S (g/kg)')
ax2.set_ylabel('Absolute salinity (g/kg)', color='steelblue', fontsize=11)
ax2.tick_params(axis='y', colors='steelblue', labelsize=10)

# ── Shade out-of-phase region ─────────────────────────────────────────────────
ax.axvspan(np.datetime64(SHADE_START), np.datetime64(SHADE_END),
           color='gold', alpha=0.2, zorder=0,
           label=f'T/S out of phase\n({SHADE_START} – {SHADE_END})')

# Vertical lines at phase transition boundaries
ax.axvline(np.datetime64(SHADE_START), color='gray',
           lw=1.2, ls='--', alpha=0.7, zorder=1)
ax.axvline(np.datetime64(SHADE_END),   color='gray',
           lw=1.2, ls='--', alpha=0.7, zorder=1)

# Transition labels at top of shaded region
ylim = ax.get_ylim()
for t_mark, label, ha in [
    (SHADE_START, 'T/S decouple', 'left'),
    (SHADE_END,   'T/S re-couple', 'right'),
]:
    ax.text(np.datetime64(t_mark), ylim[1], label,
            ha=ha, va='bottom', fontsize=8,
            color='gray', fontstyle='italic',
            transform=ax.get_xaxis_transform())

# ── x-axis formatting ─────────────────────────────────────────────────────────
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax.tick_params(axis='x', labelsize=10, rotation=30)
ax.tick_params(axis='x', which='minor', length=3, color='gray')

# ── Legend ────────────────────────────────────────────────────────────────────
shade_patch = plt.Rectangle((0, 0), 1, 1, color='gold',
                              alpha=0.3, label=f'T/S out of phase')
lines  = [l1, l2, shade_patch]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, fontsize=9, loc='upper right', framealpha=0.85)

# ── Labels ────────────────────────────────────────────────────────────────────
ax.set_xlabel('Time (2018)', fontsize=11)
ax.set_title(f'TNBD — temperature and salinity ({RESAMPLE_DAYS}-day means)',
             fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)

plt.tight_layout()
# plt.savefig('TS_outofphase_shaded.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS
# ══════════════════════════════════════════════════════════════════════════════
MOORING_NAME  = 'TNBD'
ds_ctd        = TNBD_ctd_bot
RESAMPLE_DAYS = 1

SHADE_START = '2018-09-10'
SHADE_END   = '2018-11-06'

OUT_OF_PHASE_TIME = '2018-09-10T16:40'
IN_PHASE_TIME     = '2018-11-06T11:30'

# ══════════════════════════════════════════════════════════════════════════════

# ── Prep ──────────────────────────────────────────────────────────────────────
time  = ds_ctd.time.values
theta = ds_ctd['theta'].values
absal = ds_ctd['abs_sal'].values

# Resampled for timeseries panel
rule     = f'{RESAMPLE_DAYS}D'
theta_rs = ds_ctd['theta'].resample(time=rule).mean()
sal_rs   = ds_ctd['abs_sal'].resample(time=rule).mean()
t_rs     = theta_rs.time.values

# Full resolution for T/S diagram
time_num   = mdates.date2num(time)
tmin, tmax = time_num.min(), time_num.max()

def nearest_idx(time_arr, t_str):
    return np.argmin(np.abs(time_arr - np.datetime64(t_str)))

idx_out = nearest_idx(time, OUT_OF_PHASE_TIME)
idx_in  = nearest_idx(time, IN_PHASE_TIME)

# ── FIGURE: two panels side by side ──────────────────────────────────────────
fig = plt.figure(figsize=(18, 6))
gs  = fig.add_gridspec(1, 2,
                        width_ratios=[2.5, 1],
                        wspace=0.35)

ax_ts = fig.add_subplot(gs[0, 0])   # left: full timeseries
ax_TS = fig.add_subplot(gs[0, 1])   # right: T/S diagram

# ══════════════════════════════════════════════════════════════════════════════
# PANEL A: full record timeseries with shading
# ══════════════════════════════════════════════════════════════════════════════
ax2 = ax_ts.twinx()

l1, = ax_ts.plot(t_rs, theta_rs.values,
                  color='tomato', lw=1.8, label='θ (°C)')
ax_ts.set_ylabel('Potential temperature (°C)',
                  color='tomato', fontsize=11)
ax_ts.tick_params(axis='y', colors='tomato', labelsize=10)

l2, = ax2.plot(t_rs, sal_rs.values,
               color='steelblue', lw=1.8, label='S (g/kg)')
ax2.set_ylabel('Absolute salinity (g/kg)',
               color='steelblue', fontsize=11)
ax2.tick_params(axis='y', colors='steelblue', labelsize=10)

# Shaded out-of-phase region
ax_ts.axvspan(np.datetime64(SHADE_START), np.datetime64(SHADE_END),
              color='gold', alpha=0.2, zorder=0)

# Vertical transition lines
ax_ts.axvline(np.datetime64(SHADE_START), color='gray',
              lw=1.2, ls='--', alpha=0.7, zorder=1)
ax_ts.axvline(np.datetime64(SHADE_END),   color='gray',
              lw=1.2, ls='--', alpha=0.7, zorder=1)

# Star markers on timeseries at transition times
idx_out_rs = nearest_idx(t_rs, OUT_OF_PHASE_TIME)
idx_in_rs  = nearest_idx(t_rs, IN_PHASE_TIME)

ax_ts.plot(t_rs[idx_out_rs], theta_rs.values[idx_out_rs],
           marker='*', ms=12, color='k',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6)
ax_ts.plot(t_rs[idx_in_rs], theta_rs.values[idx_in_rs],
           marker='*', ms=12, color='seagreen',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6)

# Transition labels
# ylim = ax_ts.get_ylim()
# for t_mark, label, ha in [
#     (SHADE_START, 'T/S decouple', 'left'),
#     (SHADE_END,   'T/S re-couple', 'right'),
# ]:
#     ax_ts.text(np.datetime64(t_mark), 1.01, label,
#                ha=ha, va='bottom', fontsize=8,
#                color='gray', fontstyle='italic',
#                transform=ax_ts.get_xaxis_transform())

# x-axis
ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_ts.xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax_ts.tick_params(axis='x', labelsize=10, rotation=30)
ax_ts.tick_params(axis='x', which='minor', length=3, color='gray')

# Legend
shade_patch = plt.Rectangle((0,0), 1, 1, color='gold',
                              alpha=0.3, label='T/S out of phase')
star_out    = plt.Line2D([0], [0], marker='*', color='none',
                          markerfacecolor='k',
                          markeredgecolor='white',
                          markersize=10,
                          label=f'T/S decouple ({SHADE_START})')
star_in     = plt.Line2D([0], [0], marker='*', color='none',
                          markerfacecolor='seagreen',
                          markeredgecolor='white',
                          markersize=10,
                          label=f'T/S re-couple ({SHADE_END})')
lines       = [l1, l2, shade_patch, star_out, star_in]
labels      = [l.get_label() for l in lines]
ax_ts.legend(lines, labels, fontsize=8, loc='lower left',
              framealpha=0.85)

# ax_ts.set_xlabel('Time (2018)', fontsize=11)
ax_ts.set_title(f'a) {MOORING_NAME} — T and S ({RESAMPLE_DAYS}-day means)',
                fontsize=11, fontweight='bold', loc='left')
ax_ts.spines['top'].set_visible(False)

# ══════════════════════════════════════════════════════════════════════════════
# PANEL B: T/S diagram colored by time
# ══════════════════════════════════════════════════════════════════════════════
# ax_TS.set_box_aspect(1)

sc = ax_TS.scatter(absal, theta,
                    c=time_num, cmap='plasma',
                    vmin=tmin, vmax=tmax,
                    s=2, alpha=0.5, zorder=2)

TS_template(ax_TS,
            mint=float(np.nanmin(theta)) - 0.005,
            maxt=float(np.nanmax(theta)) + 0.005,
            mins=float(np.nanmin(absal)) - 0.01,
            maxs=float(np.nanmax(absal)) + 0.01)

# Phase transition markers
ax_TS.plot(absal[idx_out], theta[idx_out],
           marker='*', ms=14, color='k',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6, label=f'T/S decouple')
ax_TS.plot(absal[idx_in], theta[idx_in],
           marker='*', ms=14, color='seagreen',
           markeredgecolor='white', markeredgewidth=0.8,
           zorder=6, label=f'T/S re-couple')

ax_TS.set_xlabel('Absolute salinity (g/kg)', fontsize=11)
ax_TS.set_ylabel('Potential temperature (°C)', fontsize=11)
ax_TS.tick_params(labelsize=10)
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)
ax_TS.set_title('b) T/S diagram — full record',
                fontsize=11, fontweight='bold', loc='left')
ax_TS.legend(fontsize=8, loc='lower right', framealpha=0.85,
              markerscale=0.8)

for lbl in ax_TS.get_xticklabels():
    lbl.set_rotation(30); lbl.set_ha('right')

# Colorbar
cbar = fig.colorbar(sc, ax=ax_TS,
                     orientation='horizontal',
                     fraction=0.04, pad=0.25, aspect=25)
cbar.ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
cbar.ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
cbar.set_label('Time (2018)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# ── Title ─────────────────────────────────────────────────────────────────────
# fig.suptitle(f'{MOORING_NAME} — seasonal T/S evolution and phase transitions',
#              fontsize=13, fontweight='bold', y=1.02)

# plt.savefig(f'TS_timeseries_diagram_{MOORING_NAME}.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
 
# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS
# ══════════════════════════════════════════════════════════════════════════════
MOORING_NAME  = 'TNBD'
ds_ctd        = TNBD_ctd_bot
RESAMPLE_DAYS = 1
 
# Define your four regions here: (start, end, color, label)
REGIONS = [
    ('2018-09-10', '2018-09-27', '#e41a1c', 'Region 1'),
    ('2018-09-27', '2018-10-14', '#ff7f00', 'Region 2'),
    ('2018-10-14', '2018-10-21', '#4daf4a', 'Region 3'),
    ('2018-10-21', '2018-11-06', '#377eb8', 'Region 4'),
]
 
# ══════════════════════════════════════════════════════════════════════════════
 
# ── Prep ──────────────────────────────────────────────────────────────────────
time  = ds_ctd.time.values
theta = ds_ctd['theta'].values
absal = ds_ctd['abs_sal'].values
 
# Resampled for timeseries panel
rule     = f'{RESAMPLE_DAYS}D'
theta_rs = ds_ctd['theta'].resample(time=rule).mean()
sal_rs   = ds_ctd['abs_sal'].resample(time=rule).mean()
t_rs     = theta_rs.time.values
 
# Full resolution for T/S diagram
time_num   = mdates.date2num(time)
 
# ── FIGURE: two panels side by side ──────────────────────────────────────────
fig = plt.figure(figsize=(18, 6))
gs  = fig.add_gridspec(1, 2,
                        width_ratios=[2.5, 1],
                        wspace=0.35)
 
ax_ts = fig.add_subplot(gs[0, 0])   # left: full timeseries
ax_TS = fig.add_subplot(gs[0, 1])   # right: T/S diagram
 
# ══════════════════════════════════════════════════════════════════════════════
# PANEL A: full record timeseries with region shading
# ══════════════════════════════════════════════════════════════════════════════
ax2 = ax_ts.twinx()
 
l1, = ax_ts.plot(t_rs, theta_rs.values,
                  color='tomato', lw=1.8, label='θ (°C)')
ax_ts.set_ylabel('Potential temperature (°C)',
                  color='tomato', fontsize=11)
ax_ts.tick_params(axis='y', colors='tomato', labelsize=10)
 
l2, = ax2.plot(t_rs, sal_rs.values,
               color='steelblue', lw=1.8, label='S (g/kg)')
ax2.set_ylabel('Absolute salinity (g/kg)',
               color='steelblue', fontsize=11)
ax2.tick_params(axis='y', colors='steelblue', labelsize=10)
 
# Shaded regions
for start, end, color, label in REGIONS:
    ax_ts.axvspan(np.datetime64(start), np.datetime64(end),
                  color=color, alpha=0.25, zorder=0)
 
# x-axis
ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_ts.xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax_ts.tick_params(axis='x', labelsize=10, rotation=30)
ax_ts.tick_params(axis='x', which='minor', length=3, color='gray')
 
# Legend (lines + one patch per region)
region_patches = [
    plt.Rectangle((0, 0), 1, 1, color=color, alpha=0.35, label=label)
    for _, _, color, label in REGIONS
]
lines  = [l1, l2] + region_patches
labels = [l.get_label() for l in lines]
ax_ts.legend(lines, labels, fontsize=8, loc='lower left',
              framealpha=0.85)
 
ax_ts.set_title(f'a) {MOORING_NAME} — T and S ({RESAMPLE_DAYS}-day means)',
                fontsize=11, fontweight='bold', loc='left')
ax_ts.spines['top'].set_visible(False)
 
# ══════════════════════════════════════════════════════════════════════════════
# PANEL B: T/S diagram — gray background, regions colored to match panel A
# ══════════════════════════════════════════════════════════════════════════════
 
# Background: everything in gray first
ax_TS.scatter(absal, theta,
              color='lightgray', s=2, alpha=0.5, zorder=1)
 
# Overlay each region in its matching color
for start, end, color, label in REGIONS:
    mask = (time >= np.datetime64(start)) & (time <= np.datetime64(end))
    ax_TS.scatter(absal[mask], theta[mask],
                  color=color, s=3, alpha=0.7, zorder=2, label=label)
 
TS_template(ax_TS,
            mint=float(np.nanmin(theta)) - 0.005,
            maxt=float(np.nanmax(theta)) + 0.005,
            mins=float(np.nanmin(absal)) - 0.01,
            maxs=float(np.nanmax(absal)) + 0.01)
 
ax_TS.set_xlabel('Absolute salinity (g/kg)', fontsize=11)
ax_TS.set_ylabel('Potential temperature (°C)', fontsize=11)
ax_TS.tick_params(labelsize=10)
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)
ax_TS.set_title('b) T/S diagram — by region',
                fontsize=11, fontweight='bold', loc='left')
ax_TS.legend(fontsize=8, loc='lower right', framealpha=0.85,
              markerscale=3)
 
for lbl in ax_TS.get_xticklabels():
    lbl.set_rotation(30); lbl.set_ha('right')
 
# plt.savefig(f'TS_timeseries_diagram_{MOORING_NAME}.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS
# ══════════════════════════════════════════════════════════════════════════════
MOORING_NAME  = 'TNBD'
ds_ctd        = TNBD_ctd_bot
RESAMPLE_DAYS = 0.5

SHADE_START = '2018-09-17'
SHADE_END   = '2018-11-06'

# ══════════════════════════════════════════════════════════════════════════════

# ── Prep ──────────────────────────────────────────────────────────────────────
time  = ds_ctd.time.values
theta = ds_ctd['theta'].values
absal = ds_ctd['abs_sal'].values

# Resampled for timeseries panel
rule     = f'{RESAMPLE_DAYS}D'
theta_rs = ds_ctd['theta'].resample(time=rule).mean()
sal_rs   = ds_ctd['abs_sal'].resample(time=rule).mean()
t_rs     = theta_rs.time.values

# Full resolution for T/S diagram
time_num = mdates.date2num(time)

# Mask for the shaded window
shade_mask = (time >= np.datetime64(SHADE_START)) & (time <= np.datetime64(SHADE_END))
tmin_shade = time_num[shade_mask].min()
tmax_shade = time_num[shade_mask].max()

# ── FIGURE: two panels side by side ──────────────────────────────────────────
fig = plt.figure(figsize=(16, 4))
gs  = fig.add_gridspec(1, 2,
                        width_ratios=[3, 1],
                        wspace=0.35)

ax_ts = fig.add_subplot(gs[0, 0])   # left: full timeseries
ax_TS = fig.add_subplot(gs[0, 1])   # right: T/S diagram

# ══════════════════════════════════════════════════════════════════════════════
# PANEL A: full record timeseries with shading
# ══════════════════════════════════════════════════════════════════════════════
ax2 = ax_ts.twinx()

l1, = ax_ts.plot(t_rs, theta_rs.values,
                  color='tomato', lw=1.8, label='θ (°C)')
ax_ts.set_ylabel('Potential temperature (°C)',
                  color='tomato', fontsize=11)
ax_ts.tick_params(axis='y', colors='tomato', labelsize=10)

l2, = ax2.plot(t_rs, sal_rs.values,
               color='steelblue', lw=1.8, label='S (g/kg)')
ax2.set_ylabel('Absolute salinity (g/kg)',
               color='steelblue', fontsize=11)
ax2.tick_params(axis='y', colors='steelblue', labelsize=10)

# Shaded out-of-phase region
ax_ts.axvspan(np.datetime64(SHADE_START), np.datetime64(SHADE_END),
              color='gold', alpha=0.2, zorder=0)

# Vertical transition lines
ax_ts.axvline(np.datetime64(SHADE_START), color='gray',
              lw=1.2, ls='--', alpha=0.7, zorder=1)
ax_ts.axvline(np.datetime64(SHADE_END),   color='gray',
              lw=1.2, ls='--', alpha=0.7, zorder=1)

# x-axis
ax_ts.xaxis.set_major_locator(mdates.MonthLocator())
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_ts.xaxis.set_minor_locator(mdates.WeekdayLocator(interval=1))
ax_ts.tick_params(axis='x', labelsize=10, rotation=30)
ax_ts.tick_params(axis='x', which='minor', length=3, color='gray')

# Legend
shade_patch = plt.Rectangle((0, 0), 1, 1, color='gold',
                              alpha=0.3, label='T/S out of phase')
lines       = [l1, l2, shade_patch]
labels      = [l.get_label() for l in lines]
ax_ts.legend(lines, labels, fontsize=8, loc='lower left',
              framealpha=0.85)

ax_ts.set_title(f'a) {MOORING_NAME} — T and S ({RESAMPLE_DAYS}-day means)',
                fontsize=11, fontweight='bold', loc='left')
ax_ts.spines['top'].set_visible(False)

# ══════════════════════════════════════════════════════════════════════════════
# PANEL B: T/S diagram — gray background, shaded window colored by time
# ══════════════════════════════════════════════════════════════════════════════

# Background: everything outside the shaded window in gray
ax_TS.scatter(absal[~shade_mask], theta[~shade_mask],
              color='lightgray', s=2, alpha=0.5, zorder=1)

# Shaded-window points, colored by time
sc = ax_TS.scatter(absal[shade_mask], theta[shade_mask],
                    c=time_num[shade_mask], cmap='plasma',
                    vmin=tmin_shade, vmax=tmax_shade,
                    s=3, alpha=0.8, zorder=2)

TS_template(ax_TS,
            mint=float(np.nanmin(theta)) - 0.005,
            maxt=float(np.nanmax(theta)) + 0.005,
            mins=float(np.nanmin(absal)) - 0.01,
            maxs=float(np.nanmax(absal)) + 0.01)

ax_TS.set_xlabel('Absolute salinity (g/kg)', fontsize=11)
ax_TS.set_ylabel('Potential temperature (°C)', fontsize=11)
ax_TS.tick_params(labelsize=10)
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)
ax_TS.set_title('b) T/S diagram ',
                fontsize=11, fontweight='bold', loc='left')

for lbl in ax_TS.get_xticklabels():
    lbl.set_rotation(30); lbl.set_ha('right')

# Colorbar (only for the shaded-window points)
cbar = fig.colorbar(sc, ax=ax_TS,
                     orientation='horizontal',
                     fraction=0.04, pad=0.25, aspect=25)
cbar.ax.xaxis.set_major_locator(mdates.AutoDateLocator())
cbar.ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
cbar.set_label('Time within shaded window (2018)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

plt.savefig(f'TS_timeseries_diagram_{MOORING_NAME}.png',
            dpi=200, bbox_inches='tight', facecolor='white')
plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')

In [ ]:
WIN1_START = ds_ctd.isel(time=0).time
WIN1_END = ds_ctd.isel(time=-1).time
fig = plt.figure(figsize=(12,4))
ax_w1 = fig.add_subplot() 
plot_dual_timeseries(ax_w1, ds_ctd,
                     WIN1_START, WIN1_END,
                     f'a) T/S going out of phase\n'
                     f'   {WIN1_START} – {WIN1_END}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── ADJUSTABLE INPUTS ─────────────────────────────────────────────────────────
ds_ctd     = TNBD_ctd_bot
T_START    = '2018-09-05'
T_END      = '2018-11-15'   # ~1 week — adjust to zoom to days of interest

# ── SLICE TO WINDOW ───────────────────────────────────────────────────────────
ds_win = ds_ctd.sel(time=slice(T_START, T_END))
t      = ds_win.time.values
temp   = ds_win['theta'].values
sal    = ds_win['abs_sal'].values

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

ax2 = ax.twinx()

# Temperature
l1, = ax.plot(t, temp, color='tomato', lw=1.5, label='θ (°C)')
ax.set_ylabel('Conservative temperature (°C)', color='tomato', fontsize=11)
ax.tick_params(axis='y', colors='tomato', labelsize=10)

# Salinity
l2, = ax2.plot(t, sal, color='steelblue', lw=1.5, label='S (g/kg)')
ax2.set_ylabel('Absolute salinity (g/kg)', color='steelblue', fontsize=11)
ax2.tick_params(axis='y', colors='steelblue', labelsize=10)

# ── x-axis: hourly or 6-hourly ticks ─────────────────────────────────────────
# For a ~1 week window use 12-hourly major ticks with daily labels
# Narrow to 1-3 days: use 6-hourly or hourly ticks
n_days = (np.datetime64(T_END) - np.datetime64(T_START)) / np.timedelta64(1, 'D')

if n_days <= 1:
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_minor_locator(mdates.MinuteLocator(interval=30))
elif n_days <= 3:
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M'))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=1))
elif n_days <= 7:
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M'))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
else:
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=12))

ax.tick_params(axis='x', labelsize=9, rotation=30)
ax.tick_params(axis='x', which='minor', length=3, color='gray')

# ── Zero reference lines ──────────────────────────────────────────────────────
ax.axhline(ax.get_ylim()[0], color='k', lw=0.3, alpha=0.3)

# ── Legend ────────────────────────────────────────────────────────────────────
lines  = [l1, l2]
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, fontsize=10, loc='upper right', framealpha=0.85)

# ── Labels ────────────────────────────────────────────────────────────────────
ax.set_xlabel('Time', fontsize=11)
ax.set_title(f'TNBD — temperature and salinity\n{T_START} to {T_END}',
             fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)

plt.tight_layout()
# plt.savefig('TS_zoom_hourly.png', dpi=200,
#             bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.animation import FuncAnimation

# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA + WINDOW TO HIGHLIGHT   <-- edit these
# ══════════════════════════════════════════════════════════════════════════════
MOORING_NAME  = 'TNBD'
ds_ctd        = TNBD_ctd_bot
RESAMPLE_DAYS = 0.5           # smoothing for the timeseries panel only

# Out-of-phase window (same as your original shading)
SHADE_START = '2018-09-17'
SHADE_END   = '2018-11-06'

# Which part of the record to animate (None = whole record)
PLAY_START = None             # e.g. '2018-08-01'
PLAY_END   = None             # e.g. '2018-12-15'

# ══════════════════════════════════════════════════════════════════════════════
# 2. HIGHLIGHT OPTIONS
# ══════════════════════════════════════════════════════════════════════════════
HIGHLIGHT_WINDOW    = True    # False = no highlighting at all, just one color
SHADE_COLOR         = 'gold'  # shaded band on the timeseries
HIGHLIGHT_PTS       = True    # draw T/S points inside the window in a
                              # different color (False = one color throughout)
PTS_COLOR           = 'dimgray'     # T/S points outside the window
PTS_HIGHLIGHT_COLOR = 'darkorange'  # T/S points inside the window

# ══════════════════════════════════════════════════════════════════════════════
# 3. ANIMATION TIMING   <-- speed knobs
# ══════════════════════════════════════════════════════════════════════════════
N_FRAMES    = 450             # frames to sweep the play range (fewer = faster)
SLOW_FACTOR = 3.0             # >1 = play the highlighted window this many times
                              # slower (more frames per day). 1 = constant speed.
HOLD_FRAMES = 40              # pause at the end
FPS         = 20              # total length = (N_FRAMES + HOLD_FRAMES) / FPS
                              # = (450 + 40) / 20 = 24.5 s
TRAIL_DAYS  = None            # None = keep all points; or a number of days to
                              # show only a trailing tail (e.g. 20)

# ══════════════════════════════════════════════════════════════════════════════
# 4. FIGURE SIZE + STYLE  (half a 13.33 x 7.5 in slide)
# ══════════════════════════════════════════════════════════════════════════════
FIG_W, FIG_H  = 6.6, 6.4      # keep FIG * DPI an even number of pixels for mp4
DPI           = 150
HEIGHT_RATIOS = [1, 1.2]      # timeseries : T/S diagram

plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'axes.linewidth':    1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

T_COLOR, S_COLOR = 'tomato', 'steelblue'
BG_LW, HL_LW     = 1.0, 2.2   # faded full-record line / drawn-so-far line
BG_DOT, PT_DOT   = 2, 6       # T/S dots: background / animated
HEAD_SIZE        = 9          # head marker on the T/S diagram

# ══════════════════════════════════════════════════════════════════════════════
# PREP DATA
# ══════════════════════════════════════════════════════════════════════════════
time     = ds_ctd.time.values
theta    = ds_ctd['theta'].values
absal    = ds_ctd['abs_sal'].values
time_num = mdates.date2num(time)

rule     = f'{RESAMPLE_DAYS}D'
theta_rs = ds_ctd['theta'].resample(time=rule).mean()
sal_rs   = ds_ctd['abs_sal'].resample(time=rule).mean()
t_rs     = mdates.date2num(theta_rs.time.values)
th_rs    = theta_rs.values
sa_rs    = sal_rs.values

w0 = mdates.date2num(np.datetime64(SHADE_START))
w1 = mdates.date2num(np.datetime64(SHADE_END))

p0 = mdates.date2num(np.datetime64(PLAY_START)) if PLAY_START else time_num.min()
p1 = mdates.date2num(np.datetime64(PLAY_END))   if PLAY_END   else time_num.max()

# Valid full-resolution samples inside the play range (sorted in time)
valid = np.where((time_num >= p0) & (time_num <= p1) &
                 np.isfinite(theta) & np.isfinite(absal))[0]
if valid.size == 0:
    raise ValueError('No valid data in the play range')
t_valid = time_num[valid]
in_win  = (time_num >= w0) & (time_num <= w1)    # full-length boolean

# Frame -> head position. Samples in the window get extra weight, so the
# animation spends more frames there (slower playback).
weights  = np.where(in_win[valid], SLOW_FACTOR if HIGHLIGHT_WINDOW else 1.0, 1.0)
cum      = np.cumsum(weights)
targets  = np.linspace(cum[0], cum[-1], N_FRAMES)
head_pos = np.clip(np.searchsorted(cum, targets), 0, valid.size - 1)
head_pos = np.concatenate([head_pos, np.full(HOLD_FRAMES, head_pos[-1])])

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax_ts, ax_TS) = plt.subplots(2, 1, figsize=(FIG_W, FIG_H), dpi=DPI,
                                   gridspec_kw={'height_ratios': HEIGHT_RATIOS},
                                   constrained_layout=True)
ax_sal = ax_ts.twinx()

# ── Timeseries: faded full record + shaded window ────────────────────────────
ax_ts.plot(t_rs, th_rs, color=T_COLOR, lw=BG_LW, alpha=0.3, zorder=1)
ax_sal.plot(t_rs, sa_rs, color=S_COLOR, lw=BG_LW, alpha=0.3, zorder=1)

if HIGHLIGHT_WINDOW:
    ax_ts.axvspan(w0, w1, color=SHADE_COLOR, alpha=0.25, zorder=0)
    for x in (w0, w1):
        ax_ts.axvline(x, color='gray', lw=1.0, ls='--', alpha=0.7, zorder=1)

ax_ts.set_ylabel('θ (°C)', color=T_COLOR)
ax_ts.tick_params(axis='y', colors=T_COLOR)
ax_sal.set_ylabel('$S_A$ (g/kg)', color=S_COLOR)
ax_sal.tick_params(axis='y', colors=S_COLOR)

ax_ts.set_xlim(t_rs.min(), t_rs.max())
ax_ts.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax_ts.set_xlabel(f'{MOORING_NAME}, 2018')
ax_ts.spines['top'].set_visible(False)
ax_sal.spines['top'].set_visible(False)

# ── T/S diagram: faded full record ───────────────────────────────────────────
ax_TS.scatter(absal, theta, color='lightgray', s=BG_DOT, alpha=0.5, zorder=1)

TS_template(ax_TS,
            mint=float(np.nanmin(theta)) - 0.005,
            maxt=float(np.nanmax(theta)) + 0.005,
            mins=float(np.nanmin(absal)) - 0.01,
            maxs=float(np.nanmax(absal)) + 0.01)

ax_TS.set_xlabel('Absolute salinity (g/kg)')
ax_TS.set_ylabel('Potential temperature (°C)')
ax_TS.spines['top'].set_visible(False)
ax_TS.spines['right'].set_visible(False)

# ── Animated artists ─────────────────────────────────────────────────────────
hl_T,  = ax_ts.plot([], [], color=T_COLOR, lw=HL_LW, zorder=3)
hl_S,  = ax_sal.plot([], [], color=S_COLOR, lw=HL_LW, zorder=3)
cursor = ax_ts.axvline(t_valid[0], color='k', lw=1.8, zorder=4)

pts_out = ax_TS.scatter([], [], color=PTS_COLOR, s=PT_DOT, alpha=0.85, zorder=3)
pts_in  = ax_TS.scatter([], [], color=PTS_HIGHLIGHT_COLOR, s=PT_DOT,
                        alpha=0.9, zorder=4)
head,   = ax_TS.plot([], [], 'o', ms=HEAD_SIZE, mfc='white', mec='k',
                     mew=1.5, zorder=6)

date_txt = ax_ts.text(0.01, 0.96, '', transform=ax_ts.transAxes, va='top',
                      fontsize=12, fontweight='bold')

EMPTY = np.empty((0, 2))

def update(frame):
    p   = head_pos[frame]
    h   = valid[p]                         # index of the head sample
    t_h = time_num[h]

    # Samples to draw on the T/S diagram (all so far, or a trailing window)
    start = 0
    if TRAIL_DAYS is not None:
        start = np.searchsorted(t_valid, t_h - TRAIL_DAYS)
    idx = valid[start:p + 1]

    if HIGHLIGHT_WINDOW and HIGHLIGHT_PTS:
        w = in_win[idx]
        pts_out.set_offsets(np.c_[absal[idx[~w]], theta[idx[~w]]])
        pts_in.set_offsets(np.c_[absal[idx[w]],  theta[idx[w]]])
    else:
        pts_out.set_offsets(np.c_[absal[idx], theta[idx]])
        pts_in.set_offsets(EMPTY)

    head.set_data([absal[h]], [theta[h]])

    # Timeseries: bold line up to the head + moving cursor
    n_rs = np.searchsorted(t_rs, t_h, side='right')
    m    = t_rs[:n_rs] >= p0
    hl_T.set_data(t_rs[:n_rs][m], th_rs[:n_rs][m])
    hl_S.set_data(t_rs[:n_rs][m], sa_rs[:n_rs][m])
    cursor.set_xdata([t_h, t_h])

    date_txt.set_text(mdates.num2date(t_h).strftime('%b %d'))
    return []

anim = FuncAnimation(fig, update, frames=len(head_pos),
                     interval=1000 / FPS, blit=False)

# ══════════════════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════════════════
# MP4 (best for PowerPoint: Insert > Video > This Device). Needs ffmpeg.
# GIF (loops automatically, but larger files and 256 colors).
SAVE_MP4 = True
SAVE_GIF = False
OUT_NAME = f'TS_animation_{MOORING_NAME}'

if SAVE_MP4:
    from matplotlib.animation import FFMpegWriter
    anim.save(f'{OUT_NAME}.mp4', writer=FFMpegWriter(fps=FPS, bitrate=4000),
              dpi=DPI, savefig_kwargs={'facecolor': 'white'})
if SAVE_GIF:
    from matplotlib.animation import PillowWriter
    anim.save(f'{OUT_NAME}.gif', writer=PillowWriter(fps=FPS),
              dpi=DPI, savefig_kwargs={'facecolor': 'white'})

# plt.show()

### Time Series Plot 

In [ ]:
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import colorsys

# ── ROLLING AVERAGE SETTINGS ────────────────────────────────────────────────
ROLL_DAYS   = 1                       # adjust this to change smoothing window
SAMPLES_PER_DAY = 48                  # 30-min sampling → 48 samples/day
ROLL_WINDOW = int(ROLL_DAYS * SAMPLES_PER_DAY)

def lighten_color(color, amount=0.5):
    """
    Lightens a color by mixing it toward white in HLS space.
    amount=0 -> unchanged, amount=1 -> white.
    """
    r, g, b = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = l + (1 - l) * amount
    return colorsys.hls_to_rgb(h, l, s)

# ── MOORING / DEPTH ORGANIZATION ────────────────────────────────────────────
# Row order: TNBD, D, DITD. Each mooring lists its depth-labeled datasets
# with a base color per mooring; deeper sensors get progressively lighter
# shades of that same color rather than a different linestyle.
MOORING_ROWS = [
    ('TNBD', {'top': TNBD_ctd_top, 'bot': TNBD_ctd_bot}, 'steelblue'),
    ('D',    {'top': D_ctd_top,    'mid': D_ctd_mid},    'tab:green'),
    ('DITD', {'top': DITD_ctd_top, 'bot': DITD_ctd_bot}, 'tomato'),
]

# Lightening amount per depth label (0 = full base color, closer to 1 = lighter)
DEPTH_SHADE = {'top': 0.0, 'mid': 0.45, 'bot': 0.45}

# ── FIGURE ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True, sharey='col',
                          constrained_layout=True)

for row_idx, (mooring_name, depths, color) in enumerate(MOORING_ROWS):
    ax_sal  = axes[row_idx, 0]
    ax_temp = axes[row_idx, 1]

    for depth_label, ds in depths.items():
        shade = DEPTH_SHADE.get(depth_label, 0.0)
        line_color = lighten_color(color, shade)

        sal_smooth  = ds.abs_sal.rolling(time=ROLL_WINDOW, center=True,
                                          min_periods=ROLL_WINDOW // 2).mean()
        temp_smooth = ds.theta.rolling(time=ROLL_WINDOW, center=True,
                                        min_periods=ROLL_WINDOW // 2).mean()

        ax_sal.plot(ds.time, sal_smooth, color=line_color, linestyle='-',
                    linewidth=1.1, label=depth_label)
        ax_temp.plot(ds.time, temp_smooth, color=line_color, linestyle='-',
                     linewidth=1.1, label=depth_label)

    ax_sal.set_ylabel('Absolute Salinity (g/kg)', fontsize=9)
    ax_temp.set_ylabel('Potential Temperature (°C)', fontsize=9)

    ax_sal.text(0.02, 0.1, mooring_name, transform=ax_sal.transAxes,
                fontsize=10, fontweight='bold', va='top', ha='left',
                color=color)

    ax_sal.legend(fontsize=8, loc='lower right', framealpha=0.85,
                  title='Depth', title_fontsize=8)
    ax_temp.legend(fontsize=8, loc='lower right', framealpha=0.85,
                   title='Depth', title_fontsize=8)

    ax_sal.grid(True, alpha=0.3)
    ax_temp.grid(True, alpha=0.3)

# Column titles
axes[0, 0].set_title(f'Salinity ({ROLL_DAYS}-day rolling mean)', fontsize=11)
axes[0, 1].set_title(f'Temperature ({ROLL_DAYS}-day rolling mean)', fontsize=11)

# Panel letters, top-left, reading order across the 3x2 grid (a-f)
panel_letters = ['a', 'b', 'c', 'd', 'e', 'f']
for ax, letter in zip(axes.flat, panel_letters):
    ax.text(0.02, 0.97, letter, transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left')

# Shared x-axis formatting on bottom row only
for ax in axes[-1, :]:
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha('right')
    # ax.set_xlabel('Date')

# plt.savefig('TS_timeseries_qualsredo.png',dpi=300,bbox_inches='tight',facecolor='white')
plt.show()

### Transport, tides, wind speed, and polynya area correlations 

In [ ]:
from TNB_tools.tides.tidal_decomposition import decompose_tidal_dataset

In [ ]:
# load in functions 
D_transport_lowpass, D_transport_highpass = decompose_tidal_dataset(
    ds_D, var_names=["v_along", "Q_Sv","F_salt", "F_heat_GW", "salinity", "temperature"], window_days=3, sampling_hours=0.5
)

In [ ]:
# load in functions 
TNBD_transport_lowpass, TNBD_transport_highpass = decompose_tidal_dataset(
    ds_TNBD, var_names=["v_along", "Q_Sv","F_salt", "F_heat_GW", "salinity", "temperature"], window_days=3, sampling_hours=0.5
)

In [ ]:
# load in functions 
DITD_transport_lowpass, DITD_transport_highpass = decompose_tidal_dataset(
    ds_DITD, var_names=["v_along", "Q_Sv","F_salt", "F_heat_GW", "salinity", "temperature"], window_days=3, sampling_hours=0.5
)

In [ ]:
# load in functions 
TNBD_lowpass, TNBD_highpass = decompose_tidal_dataset(
    TNBD_vel_bot, var_names=["u_prime","v_prime"], window_days=3, sampling_hours=0.5
)

D_lowpass, D_highpass = decompose_tidal_dataset(
    D_vel_bot, var_names=["u_prime","v_prime"], window_days=3, sampling_hours=0.5
)

DITD_lowpass, DITD_highpass = decompose_tidal_dataset(
    DITD_vel_top, var_names=["u_prime","v_prime"], window_days=3, sampling_hours=0.5
)

In [ ]:
fig = plt.figure(figsize=(12,4))

plt.plot(ds_TNBD.time,ds_TNBD.Q_Sv, label='total ')
plt.plot(TNBD_transport_highpass.time,TNBD_transport_highpass.Q_Sv,label='highpass')
plt.plot(TNBD_transport_lowpass.time,TNBD_transport_lowpass.Q_Sv,label='lowpass')
plt.legend()

In [ ]:
import matplotlib.dates as mdates

In [ ]:
# Dell 
# wspd = xr.open_dataset("C:/Users/mblan/Documents/TNB_Research/data/AWS_Manuela/AWS_Manuela_2001_2025.nc")

# # MAC 
wspd = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/data/AWS_Manuela_2001_2025.nc')

In [ ]:
# crop to 2018 
wspd = wspd.sel(time=slice('2018-03-14','2018-12-31'))

In [ ]:
wspd_daily = wspd.resample(time='1D').mean()

In [ ]:
# calculate tidal energy 

tidal_energy_TNBD = np.sqrt(TNBD_highpass.u_prime**2 + TNBD_highpass.v_prime**2)
tidal_energy_DITD = np.sqrt(DITD_highpass.u_prime**2 + DITD_highpass.v_prime**2)
tidal_energy_D = np.sqrt(D_highpass.u_prime**2 + D_highpass.v_prime**2)

In [ ]:
wspd_monthly = wspd.resample(time='MS').mean()

In [ ]:
TNBD_tide_monthly = tidal_energy_TNBD.resample(time='MS').mean()
DITD_tide_monthly = tidal_energy_DITD.resample(time='MS').mean()
D_tide_monthly = tidal_energy_D.resample(time='MS').mean()

In [ ]:
fig, ax = plt.subplots(nrows=3,ncols=1,figsize=(12,8))

# TRANSPORT 
ax[0].plot(ds_TNBD_monthly.time,ds_TNBD_monthly.Q_Sv_mean,color='steelblue')
ax[0].plot(ds_DITD_monthly.time,ds_DITD_monthly.Q_Sv_mean,color='tomato')
ax[0].plot(ds_D_monthly.time,ds_D_monthly.Q_Sv_mean,color='seagreen')

# TIDAL ENERGY 
ax[1].plot(TNBD_tide_monthly.time,TNBD_tide_monthly,color='steelblue',linestyle='--')
ax[1].plot(DITD_tide_monthly.time,DITD_tide_monthly,color='tomato',linestyle='--')
ax[1].plot(D_tide_monthly.time,D_tide_monthly,color='seagreen',linestyle='--')

# Wind Speed 
ax[2].plot(wspd_monthly.time,wspd_monthly.wind_speed)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- Style ---
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 10,
})

colors = {'TNBD': 'steelblue', 'DITD': 'tomato', 'D': 'seagreen'}
marker = 'o'
ms = 4  # marker size

fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(12, 8), sharex=True)

# ---- Panel 1: Transport ----
ax[0].axhline(0, color='k', linewidth=0.8, zorder=0)  # y=0 reference line

ax[0].plot(ds_TNBD_monthly.time, ds_TNBD_monthly.Q_Sv_mean,
           color=colors['TNBD'], marker=marker, markersize=ms, label='TNBD')
ax[0].plot(ds_DITD_monthly.time, ds_DITD_monthly.Q_Sv_mean,
           color=colors['DITD'], marker=marker, markersize=ms, label='DITD')
ax[0].plot(ds_D_monthly.time, ds_D_monthly.Q_Sv_mean,
           color=colors['D'], marker=marker, markersize=ms, label='D')

ax[0].set_ylabel('Transport (Sv)')
ax[0].set_ylim(-0.7, 0.7)  # <-- set to whatever range makes sense for your data
ax[0].legend(loc='upper right', ncol=3, frameon=False)
ax[0].set_title('(a) Monthly-mean transport', loc='left', fontweight='bold')

# ---- Panel 2: Tidal energy ----
ax[1].plot(TNBD_tide_monthly.time, TNBD_tide_monthly,
           color=colors['TNBD'], linestyle='--', marker=marker, markersize=ms, label='TNBD')
ax[1].plot(DITD_tide_monthly.time, DITD_tide_monthly,
           color=colors['DITD'], linestyle='--', marker=marker, markersize=ms, label='DITD')
ax[1].plot(D_tide_monthly.time, D_tide_monthly,
           color=colors['D'], linestyle='--', marker=marker, markersize=ms, label='D')

ax[1].set_ylabel('Tidal energy (m s$^{-1}$)')
ax[1].legend(loc='upper right', ncol=3, frameon=False)
ax[1].set_title('(b) Monthly tidal energy', loc='left', fontweight='bold')

# ---- Panel 3: Wind speed ----
ax[2].plot(wspd_monthly.time, wspd_monthly.wind_speed,
           color='slategray', marker=marker, markersize=ms, label='Wind speed')

ax[2].set_ylabel('Wind speed (m s$^{-1}$)')
ax[2].set_xlabel('Time')
ax[2].legend(loc='upper right', frameon=False)
ax[2].set_title('(c) Monthly-mean wind speed', loc='left', fontweight='bold')

# ---- Shared x-axis formatting ----
ax[2].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.autofmt_xdate()

for a in ax:
    a.grid(alpha=0.3)
    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)

plt.tight_layout()
# plt.savefig('transport_tide_wind_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

### Transport At HSSW Arrival Time 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── TIME WINDOW ───────────────────────────────────────────────────────────────
T_START = '2018-08-15'
T_END   = '2018-10-01'

# ── SLICE DATASETS TO TIME WINDOW ────────────────────────────────────────────
ctd  = TNBD_ctd_bot.sel(time=slice(T_START, T_END))
tran = ds_TNBD.sel(time=slice(T_START, T_END))

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          sharex=True,
                          gridspec_kw={'hspace': 0.08})

# ── Panel 1: Temperature and Salinity (double y-axis) ────────────────────────
ax1  = axes[0]
ax1b = ax1.twinx()

# Salinity on left axis
l1, = ax1.plot(ctd.time.values, ctd['abs_sal'].values,
               color='steelblue', lw=1.5, label='Salinity')
ax1.set_ylabel('Absolute salinity (g/kg)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', colors='steelblue', labelsize=10)

# Temperature on right axis
l2, = ax1b.plot(ctd.time.values, ctd['theta'].values,
                color='tomato', lw=1.5, label='Temperature')
ax1b.set_ylabel('Conservative temperature (°C)', color='tomato', fontsize=11)
ax1b.tick_params(axis='y', colors='tomato', labelsize=10)

# Combined legend
lines  = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, fontsize=10, loc='upper right',
           framealpha=0.8)

ax1.text(0.01, 0.92, 'Hydrography — TNBD',
         transform=ax1.transAxes, fontsize=10,
         fontweight='bold', va='top')
ax1.spines['top'].set_visible(False)
ax1b.spines['top'].set_visible(False)

# ── Panel 2: Volume transport ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(tran.time.values, tran['Q_Sv'].values,
         color='steelblue', lw=1.5)
ax2.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values < 0),
                 color='steelblue', alpha=0.15)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values > 0),
                 color='tomato', alpha=0.15)
ax2.set_ylabel('Volume transport (Sv)', fontsize=11)
ax2.text(0.01, 0.92, 'Volume transport — TNBD',
         transform=ax2.transAxes, fontsize=10,
         fontweight='bold', va='top', color='steelblue')
ax2.tick_params(axis='y', labelsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Shared x-axis formatting ───────────────────────────────────────────────────
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[-1].tick_params(axis='x', rotation=30, labelsize=10)

# plt.savefig('TNBD_hydrography_transport.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── TIME WINDOW ───────────────────────────────────────────────────────────────
T_START = '2018-08-15'
T_END   = '2018-10-01'

# ── SLICE DATASETS TO TIME WINDOW ────────────────────────────────────────────
ctd  = DITD_ctd_top.sel(time=slice(T_START, T_END))
tran = ds_DITD.sel(time=slice(T_START, T_END))

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          sharex=True,
                          gridspec_kw={'hspace': 0.08})

# ── Panel 1: Temperature and Salinity (double y-axis) ────────────────────────
ax1  = axes[0]
ax1b = ax1.twinx()

# Salinity on left axis
l1, = ax1.plot(ctd.time.values, ctd['abs_sal'].values,
               color='steelblue', lw=1.5, label='Salinity')
ax1.set_ylabel('Absolute salinity (g/kg)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', colors='steelblue', labelsize=10)

# Temperature on right axis
l2, = ax1b.plot(ctd.time.values, ctd['theta'].values,
                color='tomato', lw=1.5, label='Temperature')
ax1b.set_ylabel('Conservative temperature (°C)', color='tomato', fontsize=11)
ax1b.tick_params(axis='y', colors='tomato', labelsize=10)

# Combined legend
lines  = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, fontsize=10, loc='upper right',
           framealpha=0.8)

ax1.text(0.01, 0.92, 'Hydrography — TNBD',
         transform=ax1.transAxes, fontsize=10,
         fontweight='bold', va='top')
ax1.spines['top'].set_visible(False)
ax1b.spines['top'].set_visible(False)

# ── Panel 2: Volume transport ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(tran.time.values, tran['Q_Sv'].values,
         color='steelblue', lw=1.5)
ax2.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values < 0),
                 color='steelblue', alpha=0.15)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values > 0),
                 color='tomato', alpha=0.15)
ax2.set_ylabel('Volume transport (Sv)', fontsize=11)
ax2.text(0.01, 0.92, 'Volume transport — TNBD',
         transform=ax2.transAxes, fontsize=10,
         fontweight='bold', va='top', color='steelblue')
ax2.tick_params(axis='y', labelsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Shared x-axis formatting ───────────────────────────────────────────────────
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[-1].tick_params(axis='x', rotation=30, labelsize=10)

# plt.savefig('TNBD_hydrography_transport.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# ── TIME WINDOW ───────────────────────────────────────────────────────────────
T_START = '2018-08-15'
T_END   = '2018-10-01'

# ── SLICE DATASETS TO TIME WINDOW ────────────────────────────────────────────
ctd  = D_ctd_mid.sel(time=slice(T_START, T_END))
tran = ds_D.sel(time=slice(T_START, T_END))

# ── FIGURE ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          sharex=True,
                          gridspec_kw={'hspace': 0.08})

# ── Panel 1: Temperature and Salinity (double y-axis) ────────────────────────
ax1  = axes[0]
ax1b = ax1.twinx()

# Salinity on left axis
l1, = ax1.plot(ctd.time.values, ctd['abs_sal'].values,
               color='steelblue', lw=1.5, label='Salinity')
ax1.set_ylabel('Absolute salinity (g/kg)', color='steelblue', fontsize=11)
ax1.tick_params(axis='y', colors='steelblue', labelsize=10)

# Temperature on right axis
l2, = ax1b.plot(ctd.time.values, ctd['theta'].values,
                color='tomato', lw=1.5, label='Temperature')
ax1b.set_ylabel('Conservative temperature (°C)', color='tomato', fontsize=11)
ax1b.tick_params(axis='y', colors='tomato', labelsize=10)

# Combined legend
lines  = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, fontsize=10, loc='upper right',
           framealpha=0.8)

ax1.text(0.01, 0.92, 'Hydrography — TNBD',
         transform=ax1.transAxes, fontsize=10,
         fontweight='bold', va='top')
ax1.spines['top'].set_visible(False)
ax1b.spines['top'].set_visible(False)

# ── Panel 2: Volume transport ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(tran.time.values, tran['Q_Sv'].values,
         color='steelblue', lw=1.5)
ax2.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values < 0),
                 color='steelblue', alpha=0.15)
ax2.fill_between(tran.time.values, tran['Q_Sv'].values, 0,
                 where=(tran['Q_Sv'].values > 0),
                 color='tomato', alpha=0.15)
ax2.set_ylabel('Volume transport (Sv)', fontsize=11)
ax2.text(0.01, 0.92, 'Volume transport — TNBD',
         transform=ax2.transAxes, fontsize=10,
         fontweight='bold', va='top', color='steelblue')
ax2.tick_params(axis='y', labelsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Shared x-axis formatting ───────────────────────────────────────────────────
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[-1].tick_params(axis='x', rotation=30, labelsize=10)

# plt.savefig('TNBD_hydrography_transport.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

### current vector figure 

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from scipy.ndimage import gaussian_filter
from pyproj import Transformer

# ══════════════════════════════════════════════════════════════════════════════
# USER INPUTS
# ══════════════════════════════════════════════════════════════════════════════
MOORINGS = {
    'TNBD': {
        'lon':    float(TNBD_vel_bot.lon.values),
        'lat':    float(TNBD_vel_bot.lat.values),
        'color':  'steelblue',
        'vel_ds': TNBD_vel_bot,
        'label_offset': (0.05, 0.06),
    },
    'D': {
        'lon':    float(D_vel_bot.lon.values),
        'lat':    float(D_vel_bot.lat.values),
        'color':  'seagreen',
        'vel_ds': D_vel_bot,
        'label_offset': (-0.12, 0.06),
    },
    'DITD': {
        'lon':    float(DITD_ctd_top.lon.values),
        'lat':    float(DITD_ctd_top.lat.values),
        'color':  'tomato',
        'vel_ds': DITD_vel_top,
        'label_offset': (0.05, 0.06),
    },
}

LON_BOUNDS     = (161.5, 168.0)   # more zoomed out
LAT_BOUNDS     = (-76.2, -73.8)   # more zoomed out and square
DEPTH_CONTOURS = [500, 800, 1000, 1200]
MAP_QUIVER_SCALE  = 0.06    # map mean current arrows
TS_QUIVER_SCALE   = 0.15    # timeseries arrows — smaller = longer arrows
RESAMPLE_RULE     = 'W'

# ══════════════════════════════════════════════════════════════════════════════
# LOAD BEDMACHINE
# ══════════════════════════════════════════════════════════════════════════════
transformer_inv = Transformer.from_crs('EPSG:3031', 'EPSG:4326',
                                        always_xy=True)
x_2d, y_2d     = np.meshgrid(bath['x'].values, bath['y'].values)
lon_2d, lat_2d = transformer_inv.transform(x_2d, y_2d)

bed  = bath['bed'].values.astype(float)
mask = bath['mask'].values

depth_raw  = np.where(bed < 0, -bed, np.nan)
ocean_mask = (mask == 0) | (mask == 3)
depth_plot = np.where(ocean_mask, depth_raw, np.nan)
depth_sm   = gaussian_filter(np.where(ocean_mask, depth_raw, 0), sigma=2)
depth_sm   = np.where(ocean_mask, depth_sm, np.nan)

in_bounds      = ((lon_2d >= LON_BOUNDS[0]) & (lon_2d <= LON_BOUNDS[1]) &
                   (lat_2d >= LAT_BOUNDS[0]) & (lat_2d <= LAT_BOUNDS[1]))
depth_inbounds = depth_plot[in_bounds & np.isfinite(depth_plot)]
vmax           = np.nanpercentile(depth_inbounds, 98) \
                 if depth_inbounds.size > 0 else 2000

lat_mean = np.mean(LAT_BOUNDS)

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE VELOCITIES
# ══════════════════════════════════════════════════════════════════════════════
for name, props in MOORINGS.items():
    ds = props['vel_ds']
    props['u_mean']   = float(np.nanmean(ds['u'].values))
    props['v_mean']   = float(np.nanmean(ds['v'].values))
    props['speed']    = np.sqrt(props['u_mean']**2 + props['v_mean']**2)
    props['u_weekly'] = ds['u'].resample(time=RESAMPLE_RULE).mean()
    props['v_weekly'] = ds['v'].resample(time=RESAMPLE_RULE).mean()
    print(f"{name}: u={props['u_mean']:.3f}  v={props['v_mean']:.3f}  "
          f"speed={props['speed']:.3f} m/s")

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE LAYOUT
# ══════════════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(17, 11))

outer = gridspec.GridSpec(1, 2,
                           width_ratios=[1.1, 1],
                           wspace=0.25,             # more space between columns
                           left=0.06, right=0.96,
                           top=0.93,  bottom=0.08)

ax_map = fig.add_subplot(outer[0, 0])

ts_gs = gridspec.GridSpecFromSubplotSpec(
    3, 1,
    subplot_spec=outer[0, 1],
    hspace=0.15
)
# TNBD top, D middle, DITD bottom
ts_axes = {
    'TNBD': fig.add_subplot(ts_gs[0]),
    'D':    fig.add_subplot(ts_gs[1]),
    'DITD': fig.add_subplot(ts_gs[2]),
}

# ══════════════════════════════════════════════════════════════════════════════
# MAP PANEL
# ══════════════════════════════════════════════════════════════════════════════
pcm = ax_map.pcolormesh(lon_2d, lat_2d, depth_plot,
                         cmap='Blues', shading='auto',
                         vmin=0, vmax=vmax, zorder=1)
cbar = plt.colorbar(pcm, ax=ax_map, pad=0.02,
                     fraction=0.025, aspect=25)
cbar.set_label('Depth (m)', fontsize=9)
cbar.ax.tick_params(labelsize=8)

land = np.where((mask == 1) | (mask == 2), 1.0, np.nan)
ax_map.pcolormesh(lon_2d, lat_2d, land,
                   cmap='Greys', shading='auto',
                   vmin=0, vmax=1, alpha=0.85, zorder=2)

floating_white = np.where(mask == 3, 1.0, np.nan)
ax_map.pcolormesh(lon_2d, lat_2d, floating_white,
                   cmap='Greys_r', shading='auto',
                   vmin=0, vmax=1, alpha=0.45, zorder=3)
ax_map.contour(lon_2d, lat_2d, (mask == 3).astype(float),
               levels=[0.5], colors=['tab:red'],
               linewidths=1.2, linestyles='-', zorder=4)

valid_contours = [c for c in DEPTH_CONTOURS
                  if np.nanmin(depth_sm) < c < np.nanmax(depth_sm)]
if valid_contours:
    cs = ax_map.contour(lon_2d, lat_2d, depth_sm,
                         levels=valid_contours,
                         colors='white', linewidths=0.6,
                         alpha=0.5, zorder=5)
    ax_map.clabel(cs, levels=valid_contours[::2], fmt='%d m',
                  fontsize=7, colors='white', inline=True)

for name, props in MOORINGS.items():
    lon, lat   = props['lon'], props['lat']
    color      = props['color']
    dlon, dlat = props['label_offset']

    ax_map.plot(lon, lat, marker='o', markersize=9,
                color=color, markeredgecolor='white',
                markeredgewidth=1.2, linestyle='none', zorder=8)

    txt = ax_map.text(lon + dlon, lat + dlat, name,
                      ha='center', va='bottom', fontsize=10,
                      fontweight='bold', color=color, zorder=9)
    txt.set_path_effects([pe.withStroke(linewidth=2.5,
                                         foreground='white')])

    qv = ax_map.quiver(lon, lat,
                        props['u_mean'], props['v_mean'],
                        color=color,
                        scale=MAP_QUIVER_SCALE,
                        scale_units='inches',
                        width=0.006,
                        headwidth=4, headlength=5,
                        zorder=10)

ax_map.quiverkey(qv, X=0.76, Y=0.05,
                  U=0.05, label='0.05 m/s',
                  labelpos='E',
                  fontproperties={'size': 8},
                  color='white', labelcolor='white',
                  zorder=11)

ax_map.annotate('', xy=(0.96, 0.18), xytext=(0.96, 0.11),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='white',
                                lw=1.8, mutation_scale=14))
ax_map.text(0.96, 0.08, 'N', transform=ax_map.transAxes,
            ha='center', va='top', fontsize=9,
            color='white', fontweight='bold')

ax_map.set_aspect(1.0 / np.cos(np.deg2rad(lat_mean)))
ax_map.set_xlim(LON_BOUNDS)
ax_map.set_ylim(LAT_BOUNDS)
ax_map.set_xlabel('Longitude (°E)', fontsize=10)
ax_map.set_ylabel('Latitude (°N)', fontsize=10)
ax_map.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.1f}°E'))
ax_map.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda y, _: f'{y:.1f}°'))
ax_map.tick_params(labelsize=9)
ax_map.set_title('a) Mean circulation — Drygalski Trough 2018',
                 fontsize=11, fontweight='bold', loc='left')

# Scale bar
R          = 6371.0
km_per_deg = R * np.cos(np.deg2rad(lat_mean)) * np.pi / 180.0
bar_deg    = 25 / km_per_deg
xlim, ylim = ax_map.get_xlim(), ax_map.get_ylim()
sb_x = xlim[0] + 0.04 * (xlim[1] - xlim[0])
sb_y = ylim[0] + 0.04 * (ylim[1] - ylim[0])
ax_map.plot([sb_x, sb_x + bar_deg], [sb_y, sb_y],
            color='white', lw=2.5, solid_capstyle='butt', zorder=6)
for xpt in [sb_x, sb_x + bar_deg]:
    ax_map.plot([xpt, xpt], [sb_y - 0.005, sb_y + 0.005],
                color='white', lw=2, zorder=6)
ax_map.text(sb_x + bar_deg / 2, sb_y + 0.015, '25 km',
            ha='center', va='bottom', fontsize=8,
            color='white', fontweight='bold', zorder=6)

# ══════════════════════════════════════════════════════════════════════════════
# TIMESERIES PANELS — weekly quiver sticks
# ══════════════════════════════════════════════════════════════════════════════
panel_labels = ['b)', 'c)', 'd)']
ref_arrow_drawn = False

for (name, ax), plabel in zip(ts_axes.items(), panel_labels):
    props    = MOORINGS[name]
    color    = props['color']
    u_weekly = props['u_weekly'].values
    v_weekly = props['v_weekly'].values
    t_weekly = props['u_weekly'].time.values
    t_num    = mdates.date2num(t_weekly)

    # Baseline
    ax.axhline(0, color='k', lw=0.6, ls='-', alpha=0.3, zorder=1)

    # Quiver sticks — arrows from zero line
    y_zeros = np.zeros(len(t_num))
    qv_ts = ax.quiver(t_num, y_zeros,
                       u_weekly, v_weekly,
                       color=color,
                       scale=TS_QUIVER_SCALE,
                       scale_units='inches',
                       width=0.004,
                       headwidth=3,
                       headlength=4,
                       headaxislength=3.5,
                       zorder=3)

    # y-axis limits based on max speed
    max_spd = float(np.nanmax(np.sqrt(u_weekly**2 + v_weekly**2)))
    ax.set_ylim(-max_spd * 1.4, max_spd * 1.4)

    # Formatting
    ax.set_xlim(t_num[0] - 3, t_num[-1] + 3)
    ax.set_ylabel(f'{name}\nm/s', fontsize=9, color=color)
    ax.tick_params(axis='y', labelsize=8, colors=color)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Panel label
    ax.text(0.01, 0.93, f'{plabel} {name}',
            transform=ax.transAxes, fontsize=9,
            fontweight='bold', va='top', color=color)

    # Mean speed annotation
    mean_spd = float(np.nanmean(np.sqrt(u_weekly**2 + v_weekly**2)))
    ax.text(0.99, 0.93, f'$\\bar{{|u|}}$ = {mean_spd:.3f} m/s',
            transform=ax.transAxes, fontsize=8,
            va='top', ha='right', color=color)

    # Reference arrow on top panel only
    if not ref_arrow_drawn:
        ax.quiverkey(qv_ts, X=0.88, Y=0.15,
                      U=0.1,
                      label='0.1 m/s',
                      labelpos='S',
                      fontproperties={'size': 7},
                      zorder=5)
        ref_arrow_drawn = True

    # x-axis — label bottom panel only
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    if name == 'DITD':
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        ax.tick_params(axis='x', rotation=30, labelsize=9)
    else:
        ax.tick_params(axis='x', labelbottom=False)

# fig.suptitle('Drygalski Trough — circulation and weekly velocity 2018',
#              fontsize=13, fontweight='bold')

# plt.savefig('circulation_timeseries.png',
#             dpi=200, bbox_inches='tight', facecolor='white')
plt.show()